In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2 diptest>=0.11.0
!pip install -q flash-linear-attention==0.4.2 --no-build-isolation
!pip install -q transformers==5.15.1


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import contextlib
import logging
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    prefix_cache=None,
    h_override: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
        prefix_cache: optional pre-computed prefix cache (K/V for positions 0..S-2).
        h_override: if given, used as the hidden state of the last position at
            layer_idx INSTEAD of extracting it from a full forward. The prefix
            cache is still built from input_ids[:, :-1] (context unchanged); the
            caller is responsible for h_override being consistent with
            input_ids (e.g. h_override from the same sequence). Shapes: [1, 1, D],
            [1, D] or [D]. Device must match input_ids.device (no auto-move).
            dtype is cast to the model dtype. The tensor is cloned and detached
            (fresh leaf; the caller's tensor is never mutated).
    Returns:
        J: [K, D] Jacobian matrix (K = len(token_set), unnormalized).
    Interaction with compute_hidden_jacobian:
        compute_jacobian_for_tokens takes `h_override`; compute_hidden_jacobian
        takes `h`. Both mean "use this h instead of computing it from the
        forward", but the names differ for historical reasons (compute_hidden_
        jacobian already had `h`). With h_override=None the behavior is
        bit-identical to pre-v6.0.
    """
    model.eval()
    S = input_ids.shape[1]
    # Constraint: single sequence, no padding. The replay phase builds causal
    # masks from the cache length with attention_mask=None, so any padding
    # token in the prefix would be silently treated as valid context.
    if attention_mask is not None and not bool(attention_mask.all()):
        raise ValueError(
            "compute_jacobian_for_tokens: only unpadded single sequences are "
            "supported (the replay phase cannot honor a padding mask)."
        )
    if prefix_cache is None:
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h_override is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
        h = h.to(next(model.parameters()).dtype).detach().requires_grad_(True)
    else:
        h = _normalize_h_override(h_override, model, input_ids)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h AND the row loop below must run autograd.grad with grad mode ON,
    # even when the caller is inside torch.no_grad() (e.g. eval loops in
    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not
    # have a grad_fn" — the rows loop is part of the same enable_grad block.
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
        J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _normalize_h_override(
    h_override: torch.Tensor, model, input_ids: torch.Tensor
) -> torch.Tensor:
    """Validate + normalize h_override to a detached [1, 1, D] leaf.
    Checks (in order, before any compute): shape reducible to [1, 1, D],
    D == model hidden_size, device == input_ids.device. dtype is cast to the
    model dtype. The tensor is cloned+detached (fresh leaf; caller's tensor
    never mutated, no parasitic CopyBackwards nodes).
    """
    if h_override.dim() not in (1, 2, 3) or h_override.numel() == 0:
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    if any(s != 1 for s in h_override.shape[:-1]):
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    h = h_override.reshape(1, 1, -1)
    # D lookup: flat CausalLM has config.hidden_size; the multimodal wrapper
    # (Qwen3_5ForConditionalGeneration) exposes it on config.text_config only
    # (the outer config is a heterogeneity spec without hidden_size).
    D_expected = None
    cfg = model.config
    for cand in (getattr(cfg, "text_config", None), cfg):
        if cand is not None and hasattr(cand, "hidden_size"):
            D_expected = cand.hidden_size
            break
    if D_expected is None:
        raise ValueError(
            "h_override: cannot resolve model hidden_size "
            "(neither config nor config.text_config has it)"
        )
    D = h.shape[-1]
    if D != D_expected:
        raise ValueError(
            f"h_override: D={D} does not match model hidden_size={D_expected}"
        )
    if h.device != input_ids.device:
        raise ValueError(
            f"h_override: device {h.device} does not match "
            f"input_ids.device {input_ids.device} (no auto-move)"
        )
    model_dtype = next(model.parameters()).dtype
    # clone first, then cast: detach/clone act on the caller's values before
    # any cast, and `to(same dtype)` is a no-op, so the typical fp32 case
    # allocates exactly once (audit docs/changed_review.md, item 3).
    return h.detach().clone().to(model_dtype).requires_grad_(True)
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def last_position_logits(
    model,
    h: torch.Tensor,
    S: int,
    layer_idx: int,
    prefix_cache: DynamicCache,
) -> torch.Tensor:
    """Decode h (residual at position S-1, output of layer layer_idx) through the
    remaining layers + final norm + lm_head against a prefix cache over positions
    0..S-2 and return logits [1, V].
    This is EXACTLY the path compute_jacobian_for_tokens differentiates through,
    so with prefix_cache = _run_prefix_pass(m, ids, attn) and h =
    hidden_states[layer_idx+1][:, -1:, :] it reproduces the full-forward
    logits[:, -1, :] (KL ~ 1e-7, verified). Passing the cache of a FULL forward
    (all S positions) instead double-counts the last token — the bug this helper
    exists to prevent (audit 2026-11-05, E-11a/12a/12b)."""
    return _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    Must copy every per-layer state field, not just the obvious tensors. Both
    layer types swallow unknown constructor kwargs, so passing
    ``keys=`` / ``conv_states=`` / ``recurrent_states=`` silently discards the
    state and yields an empty cache. We therefore assign attributes directly:
    * DynamicLayer (full attention): keys / values / is_initialized / dtype /
      device. Without this, ``update()`` lazy-initialises an empty cache and
      each layer attends only to the current token.
    * LinearAttentionLayer (GDN): conv_states / recurrent_states plus the flag
      dicts (``has_previous_state`` etc.). Without these, GDN layers treat the
      new token as the first in the sequence and drop all history.
    All cloned state tensors are .detach()-ed: the prefix cache is pure data
    (built under no_grad). Detaching drops pointless CopyBackwards graph nodes
    in the enable_grad decode. The transformers 5.15 in-place cache updates
    (update_recurrent_state's copy_, causal_conv1d_update's state update —
    cudagraph static-address optimisation) are NOT fixed by the detach —
    the in-place copy_ lands on the kernel's saved input regardless. Those
    are handled by _clone_gdn_states_on_entry (clone at kernel entry so the
    read object and the in-place-written object are different objects).
    Extra unknown attributes are deep-copied as a forward-compat safety net.
    Cache OBJECT-level attributes (e.g. ``offloading`` on 5.15.1; 4.x carried
    ``_seen_tokens``) are copied from ``vars(cache)`` after the layer loop:
    a fresh ``DynamicCache()`` would reset them. On 5.15.1 ``get_seq_length()``
    is derived from ``keys.shape`` (not an object field), so the post-clone
    assert below passes on both paths — on a version that tracks sequence
    length in cache state it turns a silent Jacobian corruption into a loud
    error.
    ``copy.deepcopy(prefix_cache)`` DOES produce independent copies on 5.15.1
    (tensor values are not shared). _deep_clone_cache exists for (a) detached
    tensors (no spurious CopyBackwards in the enable_grad decode) and
    (b) robustness to per-layer attribute renames — not because deepcopy
    shares storage. Neither clone fixes the in-place write into a graph-saved
    node; that is _clone_gdn_states_on_entry's job (see its docstring).
    """
    import copy as _copy
    def _cl(t):
        return t.clone().detach() if t is not None else None
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            # Deliberately NOT a vars()-walk: full-attention state fields are
            # stable across versions, and a rename would fail LOUD
            # (AttributeError), not silently corrupt the decode. The GDN
            # branch below IS generic, because GDN state fields have
            # demonstrably drifted between versions/backends.
            nl = DynamicLayer()
            nl.keys = _cl(layer.keys)
            nl.values = _cl(layer.values)
            nl.is_initialized = layer.is_initialized
            nl.dtype = getattr(layer, "dtype", None)
            nl.device = getattr(layer, "device", None)
            for k, v in vars(layer).items():
                if k not in vars(nl):
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
        else:
            # GDN (LinearAttentionLayer). Clone every state field dynamically
            # rather than hardcoding transformers-5.15.1 attribute names
            # (number_of_states, conv_states, ...): other transformers versions
            # on other backends (e.g. the Kaggle TPU torch 2.8 image) name some
            # differently, and a hardcoded name crashes with AttributeError.
            # Tensors (or dicts of tensors) are clone+detached; scalars and
            # flag-dicts are deep-copied. This is the generic form of the old
            # field-by-field copy and behaves identically on 5.15.1.
            nl = LinearAttentionLayer()
            for k, v in vars(layer).items():
                if isinstance(v, torch.Tensor):
                    setattr(nl, k, _cl(v))
                elif isinstance(v, dict) and v and all(
                    isinstance(x, torch.Tensor) for x in v.values()
                ):
                    setattr(nl, k, {kk: _cl(xv) for kk, xv in v.items()})
                else:
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
    # Cache object-level state (offloading flag on 5.15.1; 4.x: _seen_tokens):
    # a fresh DynamicCache() would reset it. ``layers`` is rebuilt above;
    # list-valued attrs are skipped to avoid aliasing the original.
    for k, v in vars(cache).items():
        if k == "layers" or isinstance(v, list):
            continue
        setattr(new_cache, k, _copy.deepcopy(v))
    # Guard: the clone must report the same sequence length (loud, not
    # silent, on a version that tracks it in cache state). A bare `assert`
    # disappears under `python -O`, turning a fail-loud check into a silent
    # one — the class of bug this guard exists to catch (audit4 Q12).
    seq_new, seq_old = new_cache.get_seq_length(), cache.get_seq_length()
    if seq_new != seq_old:
        raise RuntimeError(
            f"cache clone lost the sequence length: {seq_new} != {seq_old} "
            "(transformers cache layout changed? the clone is not a clone)")
    return new_cache
@contextlib.contextmanager
def _clone_gdn_states_on_entry():
    """Clone GDN kernel state inputs at entry: read object != written object.
    transformers 5.15 Qwen3_5GatedDeltaNet.forward resolves the kernels as
    free variables of its own module (modeling_qwen3_5), so patching the
    module attributes works — verified on 5.15.1 (calls at modeling L457/
    L506). On
    a version where GDN forward binds the kernels by a different route
    (an alias imported into another module), the patch silently no-ops and
    the in-place autograd error below resurfaces LOUD — that is the
    detection signal; there is no silent degradation.
    The recurrent loop saves the read object (version 0) for backward.
    Afterwards update_recurrent_state does a stored.copy_(new) IN PLACE on
    the very same object (5.15 cudagraph static-address optimisation) →
    version 0→1 → "modified by an inplace operation" in any backward through
    the decode (create_graph not required; the clone is also a no-op safety
    net for create_graph=True graph retention). Cloning at kernel entry makes
    the loop's saved input the fresh clone; the in-place write lands on the
    cache's original (a different object) → no in-place on a saved node.
    Re-entrancy: nesting is a NO-OP (counter below) — the outer context's
    wrappers are already active and cloning is idempotent (clone-of-clone
    changes nothing). Overwriting mq.<kernel> from OUTSIDE during the patch
    window is user error and is NOT protected (the outer exit restores its
    captured orig, discarding the overwrite).
    If the Qwen3.5 module cannot be imported, a WARNING is logged ONCE per
    interpreter (not per decode; non-Qwen3.5 builds are fine — for Qwen3.5
    with enable_grad=True the in-place autograd error is NOT protected).
    """
    global _GDN_PATCH_DEPTH
    try:
        from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
    except Exception:
        if not getattr(_clone_gdn_states_on_entry, "_warned", False):
            logging.getLogger(__name__).warning(
                "GDN monkey-patch skipped (transformers.models.qwen3_5 not "
                "importable). If the model is Qwen3.5 with enable_grad=True, "
                "in-place autograd errors may occur.")
            _clone_gdn_states_on_entry._warned = True
        yield  # non-Qwen3.5 build: nothing to patch
        return
    _GDN_PATCH_DEPTH += 1
    if _GDN_PATCH_DEPTH > 1:
        # Nested: outer context already patched; wrappers chain, and the
        # clone is idempotent, so nothing to do here.
        yield
        _GDN_PATCH_DEPTH -= 1
        return
    orig_rec = mq.torch_recurrent_gated_delta_rule
    orig_conv = mq.causal_conv1d_update
    def rec(q, k, v, g, beta, *args, initial_state=None, output_final_state=True,
            use_qk_l2norm_in_kernel=False, **kw):
        # initial_state/output_final_state arrive as KEYWORDS in 5.15.1
        # (and may be positional on other versions, hence *args); clone the
        # bound state, forward everything else untouched. A kernel whose
        # signature drops these names falls through to **kw and fails loud.
        if initial_state is not None:
            initial_state = initial_state.clone()
        return orig_rec(q, k, v, g, beta, *args, initial_state=initial_state,
                        output_final_state=output_final_state,
                        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel, **kw)
    def conv(h, cs, *args, **kw):
        # (cs, weight, [bias, activation], ...) — positional tail is
        # version-dependent; forward everything we don't own.
        if cs is not None:
            cs = cs.clone()
        return orig_conv(h, cs, *args, **kw)
    mq.torch_recurrent_gated_delta_rule = rec
    mq.causal_conv1d_update = conv
    try:
        yield
    finally:
        mq.torch_recurrent_gated_delta_rule = orig_rec
        mq.causal_conv1d_update = orig_conv
        _GDN_PATCH_DEPTH -= 1
_GDN_PATCH_DEPTH = 0
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    # Wrapper-safe: flat CausalLM -> model.model; mm wrapper -> .language_model
    text_model = _lm_module(model)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    layer_types = _layer_types(model)
    n_layers = _n_layers(model)
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via ONE batched backward (is_grads_batched).
    Replaces the per-row autograd.grad loop (K backwards through the
    decoder tail) with a single vmap-style pass: grad_outputs=eye(K)
    gives all K rows at once, 3-15x faster on T4 (review point 5).
    Verified identical to the loop on synthetic linear + nonlinear graphs;
    on the real GDN decoder tail this is an EMPIRICAL property of the
    transformers 5.15 kernels (torch_recurrent_gated_delta_rule / causal
    conv1d update under vmap-style batched backward) — there is no
    per-row fallback, so a transformers upgrade that breaks this shows up
    as a runtime error, not silent degradation.
    """
    K = len(token_set)
    return torch.autograd.grad(
        outputs=logits[0, token_set],
        inputs=h,
        grad_outputs=torch.eye(K, device=h.device, dtype=h.dtype),
        is_grads_batched=True,
        retain_graph=False,
    )[0][:, 0, 0, :]
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = (diag(p) - pp^T) @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r · diag(1+w) · (I - xhat xhat^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    # h may arrive as [1, S, D] (full-sequence hidden_states) or [1, D]/[D].
    # The last-layer READOUT Jacobian is the FINAL-position map (it feeds the
    # final-token logits), so collapse a [1, S, D] to its last position first.
    # BUG this fixes: [1, S, D].reshape(1,-1) -> [1, S*D] -> [S*D, S*D] identity
    # (e.g. [65536, 65536] = 16 GB) -> OOM on GPU / XLA compile-abort on TPU
    # (previously misattributed to an "XLA phantom dot" shape bug).
    if h.dim() == 3:
        h = h[:, -1, :]
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
            - If h is None: h is extracted from a full forward and stays
              connected to model params (full gradient flow, as before).
            - If h is given: h is used AS-IS instead of the forward (the
              forward is skipped). A ready leaf h (requires_grad=True,
              grad_fn=None) is used IN PLACE — its .grad receives the
              backpropagated derivative d J_norm/d h. J_norm still depends
              on the decoder WEIGHTS between layer_idx and the final norm
              (the decode pass), so gradients also reach those params; only
              the params of layers <= layer_idx / embeddings are excluded.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
        h: pre-computed hidden state at layer_idx (reuse from a prior forward,
            e.g. KL-forward in exp17). Respected in BOTH create_graph paths
            (v6.0 bug-fix: the create_graph=True path previously ignored it
            and unconditionally re-extracted h from a full forward). Not
            renamed to h_override for backward compatibility; the asymmetry
            with compute_jacobian_for_tokens(h_override=...) is intentional.
    """
    model.eval()
    n_layers = _n_layers(model)
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        if h is None:
            # Full forward to get h at layer_idx — connected to all model params + hooks
            with torch.enable_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        # Normalize [1, D] / [D] -> [1, 1, D]; keep any existing grad graph intact.
        h = h.reshape(1, 1, -1) if h.dim() in (1, 2) else h
        model_dtype = next(model.parameters()).dtype
        assert h is not None  # narrowed: param was Tensor|None
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            if h.requires_grad:
                # Caller passed a ready leaf that already requires grad
                # (e.g. a representation-holonomy h_leaf). Use it IN PLACE:
                # detach() would create a NEW leaf and the caller's tensor
                # would never receive .grad. It is already a leaf (grad_fn
                # None), so no graph surgery is needed.
                pass
            else:
                # Fully frozen graph: no trainable path reaches this layer
                # (e.g. linear_attention pair with no LoRA patch, or a
                # no_grad caller), OR a disconnected h was passed in. Make h
                # a leaf so the decoder pass below still yields a
                # differentiable h_norm; J_norm values are identical to the
                # connected case.
                h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm_compat(
                    model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            if h is None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
                assert outputs.hidden_states is not None  # requested
                h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
            assert h is not None
            h = h.to(next(model.parameters()).dtype)
        return _last_layer_jacobian(h, _lm_module(model).norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = text_model.config.layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    # create_graph=True: every row must retain the graph — the caller
    # backpropagates a final loss through J_norm afterwards (exp17 bridge
    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)
    # graph after the last row, so the caller's first backward would fail
    # with "backward through the graph a second time". Memory is bounded
    # by the refresh schedule (one graph per si, freed at the next refresh
    # / pair end).
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(True if create_graph else i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J
def _lm_module(model):
    """The text model with .layers/.norm/.rotary_emb.
    Flat CausalLM: model.model IS the text model. Multimodal wrapper
    (Qwen3_5ForConditionalGeneration): model.model.language_model.
    """
    mm = model.model
    if hasattr(mm, "layers"):
        return mm
    return mm.language_model
def _n_layers(model) -> int:
    """num_hidden_layers for flat or wrapper configs."""
    cfg = model.config
    return int(
        cfg.num_hidden_layers
        if hasattr(cfg, "num_hidden_layers")
        else cfg.text_config.num_hidden_layers
    )
def _layer_types(model) -> list:
    """layer_types for flat or wrapper configs."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
def _run_decoder_to_norm_compat(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """_run_decoder_to_norm over _lm_module (flat or nested model)."""
    return _run_decoder_to_norm(_lm_module(model), prefix_cache, h, S, layer_idx)
def h_norm_at_perturbed_h(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    h: torch.Tensor,
) -> torch.Tensor:
    """h_norm after the prefix, with the l_S hidden state pinned to `h`.
    Finite-diff support (B2 spot check): call twice, h and h + eps*e_j,
    and difference the two h_norms. h shape [1, 1, D] (any device/dtype);
    the pinning is done on a copy — the input is not mutated.
    """
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.no_grad():
        return _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
def _forward_h_norm(
    model,
    layer_idx,
    input_ids,
    attention_mask,
    h: torch.Tensor | None = None,
):
    """Forward once -> (h, h_norm), h graph-connected at layer l_S.
    Shared by compute_hidden_jacobian_rows and bridge_rows_backward (A4):
    the prefix pass runs before the model forward (hook-safe), h is the
    l_S hidden state (kept in graph so a caller that patched o_proj / lm_head
    gets HVP/bridge grads from the downstream backward), h_norm = RMSNorm(h)
    (last layer) or decoder-to-norm (otherwise).
    h (optional): reuse a pre-computed l_S hidden state instead of doing a
    fresh full forward (v23.1: exp17's KL forward already produced it, so
    the refresh reuses it and only runs the cheap decoder-tail-to-norm).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.enable_grad():
        if h is None:
            # standalone: full forward to obtain h (original A4 behavior)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        assert h is not None
        h = h.to(next(model.parameters()).dtype)
        if h.grad_fn is None:
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            h_norm = _run_decoder_to_norm_compat(
                model, prefix_cache, h, S, layer_idx
            )
    return h, h_norm
def bridge_objective(cos: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """The bridge objective `Σ_k w_k (1 − cos_k)` — ONE formula for the thing that is
    OPTIMIZED and the thing that is REPORTED (audit4 M1).
    Before this, `bridge_rows_backward` minimized the unweighted mean `(1−cos).mean()`
    while `bridge._weighted_cosine_loss` reported `Σ w(1−cos)` with `w = p_L/Σp_L`, so
    ""gap improved"" was not a statement about the trained objective. Both sides call
    this function now.
    `w` is used AS GIVEN: it must be a probability vector over the rows you pass in, and
    the CALLER normalizes. Deliberate — the chunked path normalizes `p` once and then
    slices, so per-chunk losses add up to the global objective. Normalizing inside here
    would renormalize every chunk by its own weight mass: chunk-size dependent, and it
    quietly changes the objective (this is exactly what the chunk-invariance test caught).
    The objective stays a convex combination like the old mean — same scale, no LR
    retuning — but concentrated: on sharp `p` the top token carries most of the gradient,
    by design. Rows dropped by a validity mask must enter with `w = 0` (the metric masks
    them at its caller; on unmasked rows the two sides are identical).
    """
    if cos.shape != w.shape:
        raise ValueError(f"bridge_objective: cos {tuple(cos.shape)} vs w {tuple(w.shape)}")
    return (w.reshape(-1).to(cos.dtype) * (1.0 - cos)).sum()
def bridge_rows_backward(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    J_tgt_p: torch.Tensor,
    extra_loss: torch.Tensor | None = None,
    chunk_size: int = 8,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> tuple[torch.Tensor, float]:
    """v23 A4: chunked bridge backward (the v22 OOM fix).
    Forward once + J_bar once, then build each chunk of K rows and backprop
    its `bridge_objective` loss immediately (extra_loss — the KL — is
    backproped with the first chunk). Backproping per chunk frees that
    chunk's 2nd-order row-graphs at once, so peak = chunk_size + 1 (the
    shared J_bar) second-order graphs, not K. Per-chunk gradients are
    identical to a single backward on the total weighted loss
    `Σ_k (p_k/Σp)(1 − cos_k)` — the SAME functional `bridge.jacobian_gap`
    reports (audit4 M1: the objective must be the metric; it used to be the
    unweighted mean).
    Gradients reach W_rows (live lm_head + B_lh @ A_lh) and o_proj (HVP
    through h). Returns (J_S_p [K, D] detached, bridge_loss scalar).
    h (optional): reuse a pre-computed l_S hidden state (v23.1) so this
    does NOT redo the full forward — only the cheap decoder-tail-to-norm
    runs. exp17's KL forward already produced the same h.
    """
    import time as _time
    K = len(token_set)
    w_obj = p.reshape(-1) / p.reshape(-1).sum()          # metric weights, Σw = 1
    h, h_norm = _forward_h_norm(
        model, layer_idx, input_ids, attention_mask, h=h
    )
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=True,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # shared; freed by the last chunk's bwd
    # extra_loss (the KL) gets a SEPARATE backward before the chunk loop.
    # retain_graph=True keeps the shared h-graph alive for the chunks while
    # the KL's OWN forward graph (a separate forward in exp17) frees here.
    # We deliberately do NOT add it to chunk 0: in a shared-forward test/
    # wiring that would free the h-graph the chunks still need.
    if extra_loss is not None:
        extra_loss.backward(retain_graph=True)
    chunks: list[torch.Tensor] = []
    total_bridge = 0.0
    for c in range(0, K, chunk_size):
        hi = min(c + chunk_size, K)
        is_last = (c + chunk_size) >= K
        rows_c = torch.stack(
            [p[i] * (_row_grad(W_rows[i]) - J_bar) for i in range(c, hi)]
        )
        tgt_c = J_tgt_p[c:hi].float()
        cos_c = torch.nn.functional.cosine_similarity(rows_c.float(), tgt_c, dim=1)
        chunk_loss = bridge_objective(cos_c, w_obj[c:hi])
        # retain_graph on all-but-last keeps the ONE shared h forward graph
        # alive so the next chunk can still build its rows; the chunk's
        # 2nd-order row-graph is released by refcounting once we del it
        # (forward nodes stay, referenced by h / h_norm / J_bar).
        chunk_loss.backward(retain_graph=not is_last)
        total_bridge += chunk_loss.item()
        chunks.append(rows_c.detach())
        del rows_c, cos_c, chunk_loss, tgt_c
        if verbose:
            print(
                f"        bridge_rows: {hi}/{K} "
                f"({hi / max(_time.time() - start, 1e-6):.1f} chunks/s)"
            )
    return torch.cat(chunks), total_bridge
def compute_hidden_jacobian_rows(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    create_graph: bool = False,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """v23 A1: per-row J_S_p = p_i*(J_i - J_bar), no dense [D,D] J_norm.
    K-subset softmax gauge — the SAME gauge as the exp16 target (p_s =
    softmax(full_logits[token_set])) and the exp17 dense bridge
    (softmax_jacobian(W[token_set]@J_norm, p_L)):
      p = re-normalized dist over the K-subset [K], W_rows = W[token_set]
      [K, D]; J_bar = d(h_norm @ (p @ W_rows))/dh; row_i =
      p[i] * (d(h_norm @ W_rows[i])/dh - J_bar).
    h stays graph-connected (via _run_decoder_to_norm_compat, which
    routes through _lm_module so both flat CausalLM and the multimodal
    wrapper work), so a caller that patched the target layer's o_proj
    and/or lm_head gets its HVP/bridge gradients from the downstream
    loss backward. chunk_size (A4) bounds the 2nd-order graphs alive
    at once; create_graph=True retains them for a later loss backward,
    create_graph=False frees each chunk after the gradient (for cheap
    no-grad refreshes / evals).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    h, h_norm = _forward_h_norm(model, layer_idx, input_ids, attention_mask)
    return _compute_jacobian_rows_matrixfree(
        h_norm,
        h,
        W_rows,
        p,
        token_set,
        create_graph=create_graph,
        chunk_size=chunk_size,
        verbose=verbose,
    )
def _compute_jacobian_rows_matrixfree(
    h_norm: torch.Tensor,
    h: torch.Tensor,
    W_rows: torch.Tensor,
    p: torch.Tensor,
    token_set: list[int],
    create_graph: bool = True,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """J_S_p [K, D] rows, K-subset softmax gauge, no dense [D,D] J_norm.
    p [K] = re-normalized softmax over the K-subset (exp16/exp17 gauge),
    W_rows [K, D] = the selected rows. J_bar = d(h_norm @ (p @ W_rows))/dh;
    row_i = p[i]*(d(h_norm @ W_rows[i])/dh - J_bar). Identical to
    softmax_jacobian(W_rows @ J_norm, p) but with K backward passes
    instead of a retained [D,D] 2nd-order matrix (v22 OOM).
    Chunked (A4): only chunk_size row-graphs alive at once.
    """
    import time as _time
    K = len(token_set)
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=create_graph,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # [D]; p@W_rows = effective selected row
    rows = []
    for c in range(0, K, chunk_size):
        for local, i in enumerate(range(c, min(c + chunk_size, K))):
            row = p[i] * (_row_grad(W_rows[i]) - J_bar)
            rows.append(row)
        if verbose:
            done = min(c + chunk_size, K)
            el = _time.time() - start
            print(f"        J_S_p: {done}/{K} ({done / max(el, 1e-6):.1f}/s)")
    out = torch.stack(rows)
    elapsed = _time.time() - start
    if verbose:
        print(f"        J_S_p done: {K}/{K} in {elapsed:.1f}s")
    return out


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_gap(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 0.0,
) -> float:
    """Compute the Jacobian gap (alignment loss, LOWER = BETTER) between large and small models.
    Terminology (v4.0): gap = weighted (1 - cos), lower=better; bridge = weighted
    cos, higher=better; gap = 1 - bridge at equal weights. Callers reporting
    "bridge" (E-16a/b) invert: bridge = 1 - jacobian_gap(...). The legacy
    `jacobian_bridge` alias (which silently returned the GAP direction despite
    its name) was removed after being identified as an inversion trap — the
    inversion lives in the experiment kernels (kaggle_kernel/kernel-exp1*/...py),
    not here.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} (disk-cached, see _inverse_gramian)
    corrects for non-orthogonal embeddings, so identity (J_L == J_S,
    E_L == E_S) yields gap ~0.
    threshold: row mask on BOTH norms, (n1 > threshold) & (n2 > threshold).
    Default 0.0 = sentinel-only mode: only exactly-zero rows are masked
    (no decay filtering). Historical default was 1e-6; the paper's
    relative quantile tau_l = 0.01 * median_i ||J_S[i]|| (E-6 calibration,
    ~1e-4 at L18) can be passed explicitly to enable decay masking.
    Epsilon convention (docs/audit4.pdf M3): paper §2.4 writes the cosine
    denominator as ``‖J‖·‖J_target‖ + ε``; the code uses ``F.cosine_similarity``
    (divide by each norm with ``clamp_min=1e-12``). On live rows the two differ
    by ``≲1e-9`` relative; on near-zero rows the additive form is WORSE — it
    turns the cosine into an arbitrary small number, while clamping keeps the
    direction. The formula-vs-code gap is therefore documented, not closed:
    closing it would move published bridge numbers for no semantic gain.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [K, D_L] through E_L^T -> [K, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [K, V] through E_S @ G_inv -> [K, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True, map_location="cpu")
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Hashes shape/device header + the first row, the last row, and up to 9
    deterministic mid-rows (uniformly spaced, seeded from row count so the
    sample is stable across runs but covers the interior). First/last rows
    alone would collide for matrices that share boundary rows (e.g. PAD/UNK)
    but differ internally. Hashing the full 248k×1024 matrix would be
    prohibitively expensive.
    KNOWN SLOP, kept on purpose (audit4 Q4): rows 0 and V−1 are fed to the
    digest twice (once explicitly, once through ``idx``), ``device`` is part of
    the key so a CPU and a GPU run get separate files, and the torch version is
    NOT in the key (a LAPACK change could alter the pseudoinverse in the last
    bits). None of these can cause a false cache HIT — they cost bytes, not
    correctness — and changing any of them invalidates the whole on-disk
    ``results/.gramians`` cache (a full (EᵀE)⁻¹ recompute for a 248k×1024
    embedding), so the key is frozen until something actually needs it.
    """
    V = E.shape[0]
    idx = [0, V - 1]
    if V > 4:
        step = (V - 1) // 10
        idx += sorted({(i * step) % (V - 1) + 1 for i in range(1, 10)})
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    for i in sorted(set(idx)):
        data += E[i].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm <= threshold (sentinel rows).
    threshold=0.0 (default): only exactly-zero rows are masked.
    threshold>0: decay masking (absolute threshold; the paper's relative
    tau_l = 0.01*median is applied by the caller, E-6 calibration).
    """
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """The reported bridge metric (LOWER = WORSE alignment): `Σ_k w_k(1−cos_k)`,
    `w = probs/Σprobs` over the valid rows. This is THE SAME functional that
    `jacobian.bridge_objective` optimizes — one formula for objective and metric
    (audit4 M1), so ""gap improved"" means the trained loss improved.
    """
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    return bridge_objective(cos_sim, probs[mask] / probs[mask].sum()).item()


In [ ]:
import torch
def _validate(jacobians: list[torch.Tensor]) -> None:
    if not jacobians:
        raise ValueError("jacobians: empty list")
    ref = jacobians[0]
    for J in jacobians[1:]:
        if J.shape != ref.shape:
            raise ValueError(f"jacobians: shape mismatch {tuple(J.shape)} vs {tuple(ref.shape)}")
        if J.device != ref.device:
            raise ValueError(f"jacobians: device mismatch {J.device} vs {ref.device}")
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Pooling is INTRA-TRAJECTORY: all J_i must be positions of ONE sample
    (document). Cross-sample pooling is invalid — see module docstring /
    paper §15 (E-0.1); aggregate across samples at the scalar-metric level.
    Args:
        jacobians: list of N Jacobian matrices (one trajectory), each [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [min(n_components, K)] sorted descending (K < n_components
        possible in degenerate cases; in practice K = |𝒮| ≥ 32).
        eigenvectors: [min(n_components, K), K] corresponding eigenvectors.
    """
    _validate(jacobians)
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def compute_esg_per_sample(
    sample_jacobians: list[list[torch.Tensor]],
    n_components: int = 32,
) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
    """Sample-local ESG: one Gramian per sample, NO cross-sample pooling.
    Args:
        sample_jacobians: list over samples; each entry is a list of the
            sample's per-position Jacobians [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: list [n_samples] of [n_components].
        eigenvectors: list [n_samples] of [n_components, K].
    Cross-sample aggregation happens on scalars (lambda, PR, SNR, r_eff),
    never on the matrices (E-0.1: cross-sample J's are near-orthogonal).
    """
    if not sample_jacobians:
        raise ValueError("sample_jacobians: empty list")
    eigenvalues = []
    eigenvectors = []
    for Js in sample_jacobians:
        ev, evec = compute_esg(Js, n_components=n_components)
        eigenvalues.append(ev)
        eigenvectors.append(evec)
    return eigenvalues, eigenvectors
def cross_positional_gramian(
    J_t: torch.Tensor,
    J_tp: torch.Tensor,
) -> torch.Tensor:
    """Cross-positional Gramian Δ_U(t, t') = J_t @ J_tp.T.
    Args:
        J_t:  [K, D] gauge-free Jacobian at position t.
        J_tp: [K, D] gauge-free Jacobian at position t'.
    Returns:
        [K, K] Gramian (symmetric iff t == t').
    Note:
        - NOT a pooling operation: each call uses exactly two positions.
          See module docstring for intra-trajectory pooling (compute_esg)
          and cross-sample pooling validity (E-0.1).
        - When t == t', result equals the inner-product part of
          compute_esg([J_t]) for N=1: compute_esg([J_t])[0][0] ==
          eigenvalue_0(Δ_U(t, t)). Checked in selfcheck.
        - Caller must ensure J_t, J_tp come from the same model / edit /
          token_set; the function does not check semantic compatibility,
          only shape / device / dtype (via _validate).
    """
    _validate([J_t, J_tp])
    return J_t.float() @ J_tp.float().T
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)  (intra-trajectory pool)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
import hashlib
import json
import os
import re
import warnings
from collections import Counter
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
# --- закрепление пула (PLAN «Запрос 11»: все результаты воспроизводимы) -------------------
# Порядок streaming-потока на Hub ничем не закреплён, поэтому «первые подходящие строки»
# меняются от прогона к прогону: E-0.1 при идентичных params дал другой top-token набор
# (pk_hash 5c12c174… -> 30d82ade…) и сдвинул C_b(sigma_d) до x31. Пин = сырые тексты принятых
# строк + отпечаток; при наличии пина поток не читается вообще, значит прогон воспроизводим
# и офлайн. Тексты (не тензоры): input_ids по-прежнему производит токенизатор.
PIN_DIR_ENV = "PRISM_CALIB_PIN_DIR"
PIN_WRITE_ENV = "PRISM_CALIB_PIN_WRITE"      # =1: разрешить записать новый пин
PIN_REQUIRE_ENV = "PRISM_CALIB_REQUIRE_PIN"  # =1: отсутствие пина = громкий отказ
LAST_POOL_FPS: dict[str, str] = {}           # domain -> отпечаток последнего загруженного пула
def pool_fingerprint(texts: list[str]) -> str:
    """Отпечаток пула по сырым текстам (от токенизатора не зависит)."""
    h = hashlib.sha256()
    for t in texts:
        h.update(t.encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()[:16]
def _default_pin_dir() -> Path | None:
    """Каталог пинов по умолчанию, или None если искать негде.
    В flattening-Kaggle-ядре модулей нет на диске и `__file__` не определён (упавший exp3:
    `NameError: name '__file__' is not defined` прямо в `_pin_path`) — значит путь репозитория
    строится только когда модуль реально лежит в репозитории; иначе пин ищется по `PIN_DIR_ENV`,
    а без него — поток (или громкий отказ при `PIN_REQUIRE_ENV`).
    """
    try:
        here = Path(__file__).resolve()
    except NameError:
        return None
    return here.parent.parent / "data" / "calib_pins"
def _pin_path(domain, model_name, n_samples, min_length, max_length,
              save_offset_mapping) -> Path | None:
    root = os.environ.get(PIN_DIR_ENV)
    base = Path(root) if root else _default_pin_dir()
    if base is None:
        return None
    slug = re.sub(r"[^0-9A-Za-z.]+", "_", model_name)
    mode = "offset" if save_offset_mapping else "plain"
    return base / f"{domain}__{slug}__n{n_samples}__min{min_length}__max{max_length}__{mode}.json"
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable, with a
    RuntimeWarning: a fallback run silently poisons cross-domain comparisons
    (E-13/E-27 T_domain), so the substitution must be visible in the log.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/
        # gated dataset would otherwise pass the check and fail inside
        # load_calibration_data (loud crash instead of the intended fallback).
        return ds_id, cfg, split, field
    except Exception:
        warnings.warn(
            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "
            f"Domain labels in this run are INVALID for cross-domain "
            f"comparisons (E-13/E-27).",
            RuntimeWarning,
            stacklevel=2,
        )
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 64,
    min_length: int = 64,
    n_samples: int = 4,
    domain: str = "wiki",
    save_offset_mapping: bool = False,
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,
    N=4); experiments pass explicit lengths. domain: 'wiki' (C4), 'math'
    (OpenMathInstruct-2), 'code' (apex-agents), 'chat' (ultrachat). Falls back
    to wiki if the target dataset is unavailable. save_offset_mapping: adds
    `offset_mapping` to each sample (E-31 sentence boundaries).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    pin = _pin_path(domain, model_name, n_samples, min_length, max_length, save_offset_mapping)
    texts: list[str] | None = None
    ds_id = cfg = split = field = None
    if pin is not None and pin.exists():
        meta = json.loads(pin.read_text(encoding="utf-8"))
        texts = list(meta["texts"])
        print(f"[calib] {domain}: пин {pin.name}, строк={len(texts)}, "
              f"pool_fp={meta['fp']}, поток не читается", flush=True)
    elif os.environ.get(PIN_REQUIRE_ENV) == "1":
        raise RuntimeError(
            f"нет пина пула {pin or f'каталог пинов недоступен — задайте {PIN_DIR_ENV}'} при "
            f"{PIN_REQUIRE_ENV}=1: прогон на потоке с Hub недопустим, "
            f"иначе числа не воспроизводимы. Сгенерируйте пин с {PIN_WRITE_ENV}=1 один раз и "
            f"закоммитьте его."
        )
    else:
        ds_id, cfg, split, field = _load_dataset_for_domain(domain)
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    # NOTE (audit4 Q9): the offset-mapping path tokenizes with
    # add_special_tokens=False, the plain path with the tokenizer default (True).
    # Deliberate, not a bug: offset_mapping must stay aligned 1:1 with the text
    # span (special tokens carry no span), and the two paths feed different
    # consumers. Consequence to keep in mind: a token set / calibration built
    # through the offset path is NOT the same sequence as the mainline E-16
    # calibration of the same text — do not mix their token sets.
    accepted: list[str] = []
    # Из пина идут уже извлечённые тексты (извлечение = _extract_text_from_sample на момент
    # записи пина); с потока идут сырые строки.
    rows = ((t, None) for t in texts) if texts is not None else ((None, raw) for raw in dataset)
    for pinned_text, raw in rows:
        text = pinned_text if pinned_text is not None else _extract_text_from_sample(raw, field)
        if not text:
            continue
        if save_offset_mapping:
            # Токенизатор вызывается на ОДНОЙ строке, поэтому input_ids — плоский список id, а
            # offset_mapping — список пар; индексация `[0]` здесь была багом (len(int) падал на
            # первом же вызове, найденном E-45b).
            toks = tokenizer(
                text, max_length=max_length, truncation=True,
                add_special_tokens=False, return_offsets_mapping=True,
            )
            if len(toks["input_ids"]) >= min_length:
                accepted.append(text)
                samples.append({
                    "input_ids": toks["input_ids"],
                    "offset_mapping": torch.tensor(
                        toks["offset_mapping"], dtype=torch.int32
                    ),
                })
        else:
            tokens = tokenizer(
                text, max_length=max_length, truncation=True,
                return_tensors="pt",
            )
            if tokens["input_ids"].shape[1] >= min_length:
                accepted.append(text)
                samples.append(tokens)
        if len(samples) >= n_samples:
            break
    if texts is None:
        texts = accepted
        fp = pool_fingerprint(texts)
        print(f"[calib] {domain}: поток с Hub, строк={len(texts)}, pool_fp={fp}", flush=True)
        if os.environ.get(PIN_WRITE_ENV) == "1":
            if pin is None:
                raise RuntimeError(
                    f"{PIN_WRITE_ENV}=1, но каталога пинов нет: {PIN_DIR_ENV} не задан, а "
                    "`__file__` недоступен (flattening-ядро). Задайте PRISM_CALIB_PIN_DIR "
                    "на путь, который попадает в вывод ядра."
                )
            pin.parent.mkdir(parents=True, exist_ok=True)
            pin.write_text(
                json.dumps(
                    {
                        "protocol": {"domain": domain, "model": model_name, "n_samples": n_samples,
                                     "min_length": min_length, "max_length": max_length,
                                     "offset_mapping": bool(save_offset_mapping)},
                        "source": {"dataset": ds_id, "config": cfg, "split": split, "field": field},
                        "fp": fp,
                        "texts": texts,
                    },
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            print(f"[calib] {domain}: пин записан -> {pin}", flush=True)
    else:
        fp = pool_fingerprint(texts)
        if len(samples) < n_samples:
            raise RuntimeError(
                f"пин {pin.name} дал {len(samples)} строк вместо {n_samples}: токенизатор или "
                f"фильтр длины изменились относительно момента записи пина (pool_fp={fp}). "
                f"Молча читать другой пул нельзя — числа стали бы невоспроизводимыми."
            )
    LAST_POOL_FPS[domain] = fp
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Frequency token set: the K tokens that most often appear in the per-sample
    top-K of the LAST position (`logits[:, -1]`) across the calibration pool.
    One basis for every position, so a stored p-row is SUB-stochastic (~half of the
    exp27b positions carry < 0.5 of the softmax mass) — that basis is what all
    published Jacobian numbers sit on. The positional convention (each position gets
    its own top-K, so p is a real truncation) is produced by
    `experiments/exp27b_dense.py: column_mode="positional"` (cache exp27c), not here;
    "union top-K at fixed K" is not definable, since the union of per-position top-K
    exceeds K (audit4 Q10 verdict round). Never mix numbers from the two bases:
    the basis belongs in the ChannelTag slice.
    """
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float32
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention.
    dtype defaults to float32: the tested protocols run on CPU, where fp16
    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.
    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller's
    setup, not here.
    """
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model.
    The bridge projects through the AN-embedding matrix W (§3/§5). For the
    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint
    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection
    would silently use the wrong matrix. We therefore warn ONCE per model if
    lm_head.weight is not the same storage as embed_tokens.weight — a loud,
    non-fatal signal that the W = E assumption does not hold for this model.
    """
    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]
    _warn_if_not_tied(model, emb)
    return emb
def _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:
    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)
    if lm_head is None:
        return
    if not lm_head.data_ptr() == emb.data_ptr():
        # data_ptr identity covers tied weights; fall back to value equality
        # for the (rare) case of a tied-but-copied view.
        if not torch.equal(lm_head.detach(), emb):
            warnings.warn(
                f"extract_embeddings: lm_head.weight is NOT tied to "
                f"embed_tokens.weight (model={type(model).__name__}). The "
                f"bridge projects through E assuming W = E (§3); with untied "
                f"weights the an-embedding matrix is lm_head.weight and the "
                f"gap/bridge numbers are silently wrong. Extract lm_head "
                f"explicitly if this is intended.",
                UserWarning,
                stacklevel=2,
            )


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
from __future__ import annotations
def _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:
    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).
    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)
    so this module stays standalone for kernel inlining.
    """
    return {i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)}
def get_layer_types(model) -> list[str]:
    """Per-layer attention type from model.config.layer_types.
    Qwen3.5 style hybrid: entries 'full_attention' or 'linear_attention'.
    `model.config` (post-AutoModel load) exposes `layer_types` at top level
    (Qwen3_5TextConfig). Returns lowercase list.
    """
    types = getattr(model.config, "layer_types", None)
    if types is None:
        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).
        tc = getattr(model.config, "text_config", None)
        types = getattr(tc, "layer_types", None)
    if types is None:
        raise RuntimeError(
            "model.config has no layer_types (and no text_config.layer_types)"
        )
    return [str(t) for t in types]
def select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:
    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""
    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:
        raise ValueError(
            f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}"
        )
    mapping = _uniform_layer_mapping(n_L, n_S)
    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}
    selected = {0, n_S - 1} | full_S
    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]
    if (n_L - 1, n_S - 1) not in pairs:
        pairs.append((n_L - 1, n_S - 1))
    return pairs
def _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:
    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""
    best_l, best_d = 0, float("inf")
    for li in range(len(mapping)):
        d = abs(mapping[li] - j)
        if d < best_d:
            best_l, best_d = li, d
    return (best_l, j)
def classify_pair(l_S: int, layer_types: list[str]) -> str:
    """'bridge' if small-model layer l_S has o_proj (full_attention), else 'kl_only'."""
    return (
        "bridge"
        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"
        else "kl_only"
    )


In [ ]:
# ---------- pinned calibration pool: wiki ----------
import json as _json, os as _os, pathlib as _pathlib
_PIN = {'protocol': {'domain': 'wiki', 'model': 'Qwen/Qwen3.5-0.8B', 'n_samples': 8, 'min_length': 32, 'max_length': 64, 'offset_mapping': False}, 'source': {'dataset': 'allenai/c4', 'config': 'en', 'split': 'train', 'field': 'text'}, 'fp': 'b786760ba3d93547', 'texts': ['Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'Discussion in \'Mac OS X Lion (10.7)\' started by axboi87, Jan 20, 2012.\nI\'ve got a 500gb internal drive and a 240gb SSD.\nWhen trying to restore using disk utility i\'m given the error "Not enough space on disk ____ to restore"\nBut I shouldn\'t have to do that!!!\nAny ideas or workarounds before resorting to the above?\nUse Carbon Copy Cloner to copy one drive to the other. I\'ve done this several times going from larger HDD to smaller SSD and I wound up with a bootable SSD drive. One step you have to remember not to skip is to use Disk Utility to partition the SSD as GUID partition scheme HFS+ before doing the clone. If it came Apple Partition Scheme, even if you let CCC do the clone, the resulting drive won\'t be bootable. CCC usually works in "file mode" and it can easily copy a larger drive (that\'s mostly empty) onto a smaller drive. If you tell CCC to clone a drive you did NOT boot from, it can work in block copy mode where the destination drive must be the same size or larger than the drive you are cloning from (if I recall).\nI\'ve actually done this somehow on Disk Utility several times (booting from a different drive (or even the dvd) so not running disk utility from the drive your cloning) and had it work just fine from larger to smaller bootable clone. Definitely format the drive cloning to first, as bootable Apple etc..\nThanks for pointing this out. My only experience using DU to go larger to smaller was when I was trying to make a Lion install stick and I was unable to restore InstallESD.dmg to a 4 GB USB stick but of course the reason that wouldn\'t fit is there was slightly more than 4 GB of data.', 'Foil plaid lycra and spandex shortall with metallic slinky insets. Attached metallic elastic belt with O-ring. Headband included. Great hip hop or jazz dance costume. Made in the USA.', "How many backlinks per day for new site?\nDiscussion in 'Black Hat SEO' started by Omoplata, Dec 3, 2010.\n1) for a newly created site, what's the max # backlinks per day I should do to be safe?\n2) how long do I have to let my site age before I can start making more blinks?\nI did about 6000 forum profiles every 24 hours for 10 days for one of my sites which had a brand new domain.\nThere is three backlinks for every of these forum profile so thats 18 000 backlinks every 24 hours and nothing happened in terms of being penalized or sandboxed. This is now maybe 3 months ago and the site is ranking on first page for a lot of my targeted keywords.\nbuild more you can in starting but do manual submission and not spammy type means manual + relevant to the post.. then after 1 month you can make a big blast..\nWow, dude, you built 18k backlinks a day on a brand new site? How quickly did you rank up? What kind of competition/searches did those keywords have?", 'The Denver Board of Education opened the 2017-18 school year with an update on projects that include new construction, upgrades, heat mitigation and quality learning environments.\nWe are excited that Denver students will be the beneficiaries of a four year, $572 million General Obligation Bond. Since the passage of the bond, our construction team has worked to schedule the projects over the four-year term of the bond.\nDenver voters on Tuesday approved bond and mill funding measures for students in Denver Public Schools, agreeing to invest $572 million in bond funding to build and improve schools and $56.6 million in operating dollars to support proven initiatives, such as early literacy.\nDenver voters say yes to bond and mill levy funding support for DPS students and schools. Click to learn more about the details of the voter-approved bond measure.\nDenver voters on Nov. 8 approved bond and mill funding measures for DPS students and schools. Learn more about what’s included in the mill levy measure.', 'BANGALORE CY JUNCTION SBC to GONDIA JUNCTION G train timings, routes, stops, and complete info.\nAs of now, 1 trains run between from BANGALORE CY JUNCTION (YPR) to GONDIA JUNCTION (G).\nThe fastest train from BANGALORE CY JUNCTION (YPR) to GONDIA JUNCTION (G) is YPR KRBA WAINGANGA EXP (12251) that departs at 23:40 and arrives to at 21:15. It takes approximately 21:35 hours.', 'I thought I was going to finish the 3rd season of the Wire tonight.\nBut there was a commentary on episode 11, so I had to re-watch Middle Ground with the commentary. Hopefully I can finish the season next weekend.', "The rich get richer and the poor get poorer eh?\nOr is it the rich think different and play by a different set of rules?\nDo the rich take responsibility and action?\nPoor people believe 'Life happens to me.' Rich people are committed to be rich.\nPoor people WANT to be rich. Rich people think big.\nPoor people think small. Rich people focus on opportunities.\nPoor people focus on obstacles. Rich people are willing to promote themselves and their value.\nPoor people think negatively about selling and promotion.\nPoor people are closed to new ideas..\nDo You think rich or poor?"]}
_pin_dir = _os.environ.get('PRISM_CALIB_PIN_DIR', '/kaggle/working/calib_pins')
_pathlib.Path(_pin_dir).mkdir(parents=True, exist_ok=True)
_pathlib.Path(_pin_dir, 'wiki__Qwen_Qwen3.5_0.8B__n8__min32__max64__plain.json').write_text(
    _json.dumps(_PIN, ensure_ascii=False), encoding='utf-8')
_os.environ['PRISM_CALIB_PIN_DIR'] = _pin_dir
_os.environ['PRISM_CALIB_REQUIRE_PIN'] = '1'
print('[calib-pin] wiki: fp=' + _PIN['fp'] + ' n=' + str(len(_PIN['texts'])) + ' ключ=' + str(_PIN['protocol']))


In [ ]:
# ---------- pinned calibration pool: wiki ----------
import json as _json, os as _os, pathlib as _pathlib
_PIN = {'protocol': {'domain': 'wiki', 'model': 'Qwen/Qwen3.5-0.8B', 'n_samples': 8, 'min_length': 64, 'max_length': 64, 'offset_mapping': False}, 'source': {'dataset': 'allenai/c4', 'config': 'en', 'split': 'train', 'field': 'text'}, 'fp': '94bb5152636f5e0b', 'texts': ['Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'Discussion in \'Mac OS X Lion (10.7)\' started by axboi87, Jan 20, 2012.\nI\'ve got a 500gb internal drive and a 240gb SSD.\nWhen trying to restore using disk utility i\'m given the error "Not enough space on disk ____ to restore"\nBut I shouldn\'t have to do that!!!\nAny ideas or workarounds before resorting to the above?\nUse Carbon Copy Cloner to copy one drive to the other. I\'ve done this several times going from larger HDD to smaller SSD and I wound up with a bootable SSD drive. One step you have to remember not to skip is to use Disk Utility to partition the SSD as GUID partition scheme HFS+ before doing the clone. If it came Apple Partition Scheme, even if you let CCC do the clone, the resulting drive won\'t be bootable. CCC usually works in "file mode" and it can easily copy a larger drive (that\'s mostly empty) onto a smaller drive. If you tell CCC to clone a drive you did NOT boot from, it can work in block copy mode where the destination drive must be the same size or larger than the drive you are cloning from (if I recall).\nI\'ve actually done this somehow on Disk Utility several times (booting from a different drive (or even the dvd) so not running disk utility from the drive your cloning) and had it work just fine from larger to smaller bootable clone. Definitely format the drive cloning to first, as bootable Apple etc..\nThanks for pointing this out. My only experience using DU to go larger to smaller was when I was trying to make a Lion install stick and I was unable to restore InstallESD.dmg to a 4 GB USB stick but of course the reason that wouldn\'t fit is there was slightly more than 4 GB of data.', "How many backlinks per day for new site?\nDiscussion in 'Black Hat SEO' started by Omoplata, Dec 3, 2010.\n1) for a newly created site, what's the max # backlinks per day I should do to be safe?\n2) how long do I have to let my site age before I can start making more blinks?\nI did about 6000 forum profiles every 24 hours for 10 days for one of my sites which had a brand new domain.\nThere is three backlinks for every of these forum profile so thats 18 000 backlinks every 24 hours and nothing happened in terms of being penalized or sandboxed. This is now maybe 3 months ago and the site is ranking on first page for a lot of my targeted keywords.\nbuild more you can in starting but do manual submission and not spammy type means manual + relevant to the post.. then after 1 month you can make a big blast..\nWow, dude, you built 18k backlinks a day on a brand new site? How quickly did you rank up? What kind of competition/searches did those keywords have?", 'The Denver Board of Education opened the 2017-18 school year with an update on projects that include new construction, upgrades, heat mitigation and quality learning environments.\nWe are excited that Denver students will be the beneficiaries of a four year, $572 million General Obligation Bond. Since the passage of the bond, our construction team has worked to schedule the projects over the four-year term of the bond.\nDenver voters on Tuesday approved bond and mill funding measures for students in Denver Public Schools, agreeing to invest $572 million in bond funding to build and improve schools and $56.6 million in operating dollars to support proven initiatives, such as early literacy.\nDenver voters say yes to bond and mill levy funding support for DPS students and schools. Click to learn more about the details of the voter-approved bond measure.\nDenver voters on Nov. 8 approved bond and mill funding measures for DPS students and schools. Learn more about what’s included in the mill levy measure.', 'BANGALORE CY JUNCTION SBC to GONDIA JUNCTION G train timings, routes, stops, and complete info.\nAs of now, 1 trains run between from BANGALORE CY JUNCTION (YPR) to GONDIA JUNCTION (G).\nThe fastest train from BANGALORE CY JUNCTION (YPR) to GONDIA JUNCTION (G) is YPR KRBA WAINGANGA EXP (12251) that departs at 23:40 and arrives to at 21:15. It takes approximately 21:35 hours.', "The rich get richer and the poor get poorer eh?\nOr is it the rich think different and play by a different set of rules?\nDo the rich take responsibility and action?\nPoor people believe 'Life happens to me.' Rich people are committed to be rich.\nPoor people WANT to be rich. Rich people think big.\nPoor people think small. Rich people focus on opportunities.\nPoor people focus on obstacles. Rich people are willing to promote themselves and their value.\nPoor people think negatively about selling and promotion.\nPoor people are closed to new ideas..\nDo You think rich or poor?", 'Biomedics 1 Day Extra are daily replacement disposable contact lenses by CooperVision Hydron. Buy one box of 90 lenses.\nBiomedics 1 Day Extra contacts give you all the convenience of a daily disposable lens with no need for solutions, cases or cleaning and are perfect for the occasional wear. These lenses have greater comfort handling with superior ease of insertion and removal.\nBiomedic 1 Day Extra are also marketed under various other brand names including Clear Choice 1-day, Ascend 1-day, easyvision CLARISION SPHERE, Clearsight 1 Day and ProView Daily Disposable.', 'Sysco Corp. has terminated its planned $3.5 billion takeover of US Foods, it announced Monday, after a federal judge blocked the combination. The company is opting instead to add $3 billion to its stock-buyback program.\nWith the deal breaking up, Sysco will pay a $300 million termination fee to US Foods and a $12.5 million fee to Performance Food Group, which had agreed to buy some US Foods facilities. Sysco, based in Houston, plans to make the share repurchases over the next two years.\nRosemont, Illinois-based US Foods operates a major distribution center in Fishers. Sysco has a large warehouse at 4000 W. 62nd St., in Indianapolis.\nSysco had fought for more than a year to gain government approval for the transaction, which antitrust regulators said would hurt competition and lead to higher prices. Sysco and US Foods dominate a market known as broadline foodservice, which supplies school cafeterias, restaurants and hotels. Sysco had argued that the acquisition would bring $1 billion in savings, letting it offer lower prices to customers.\nInvestors have responded with relief to the deal’s demise, reflecting concerns about the company undertaking an ambitious merger. Sysco shares rose 3.1 percent the day the transaction was halted by U.S. District Judge Amit Mehta, and the stock climbed again Monday morning after the merger was withdrawn.\nMehta blocked the merger on June 23 when he granted a Federal Trade Commission request to delay the transaction. The FTC had sued the companies in February, saying the deal would give Sysco an oversized share of an industry where it’s already the biggest player.\nIn arguments before Mehta in May, the two sides clashed over the scope of the market in which the companies compete. Sysco and US Foods argued that the commission was relying on a “tortured” analysis, ignoring the variety of distribution channels available to customers.\nSysco said on Monday that it weighed embarking on an appeal but decided against it.\nShares of Sysco rose as much as 1.6 percent to $39 in early trading. The stock had slid 3.3 percent this year through the end of last week.']}
_pin_dir = _os.environ.get('PRISM_CALIB_PIN_DIR', '/kaggle/working/calib_pins')
_pathlib.Path(_pin_dir).mkdir(parents=True, exist_ok=True)
_pathlib.Path(_pin_dir, 'wiki__Qwen_Qwen3.5_0.8B__n8__min64__max64__plain.json').write_text(
    _json.dumps(_PIN, ensure_ascii=False), encoding='utf-8')
_os.environ['PRISM_CALIB_PIN_DIR'] = _pin_dir
_os.environ['PRISM_CALIB_REQUIRE_PIN'] = '1'
print('[calib-pin] wiki: fp=' + _PIN['fp'] + ' n=' + str(len(_PIN['texts'])) + ' ключ=' + str(_PIN['protocol']))


In [ ]:
# ---------- pinned calibration pool: math ----------
import json as _json, os as _os, pathlib as _pathlib
_PIN = {'protocol': {'domain': 'math', 'model': 'Qwen/Qwen3.5-0.8B', 'n_samples': 8, 'min_length': 64, 'max_length': 64, 'offset_mapping': False}, 'source': {'dataset': 'nvidia/OpenMathInstruct-2', 'config': None, 'split': 'train', 'field': 'problem'}, 'fp': '1c01d1447c7018a7', 'texts': ['Ava is planning a camping trip with her friends. She wants to make sure they have enough granola bars for snacks. There will be five people total: Ava, her two friends, and her parents. They will spend 3 days and 2 nights at the campsite, and they plan to have 2 granola bars per person for breakfast and 1 granola bar per person for an afternoon snack each day. How many granola bars will Ava need to pack in total for the entire trip?', 'A farmer has 100 meters of fencing and wants to enclose a rectangular garden with an area of 500 square meters. If the farmer also wants to leave a 5-meter wide path around the entire garden, how many meters of fencing will be needed to enclose the garden and the path? Express your answer as a decimal to the nearest tenth.', 'If $f(x)$ and $g(x)$ are two rational functions such that the vertical asymptotes of $f(x)$ are canceled by the zeros of $g(x)$, and the degree of the denominator of $f(x) + g(x)$ is 2, what is the degree of the numerator of $g(x)$?', 'The owner of a parking lot wants to label each parking space with a unique four-digit code using the digits 0 through 9. The code for each space must start with a non-zero digit, and no digit can be repeated within a code. If the owner wants to label 300 spaces, how many packages of digits must be purchased, assuming each package contains 10 of each digit?', 'In base $b$, there are exactly one hundred three-digit numbers whose digits are all distinct. (That\'s "one hundred" in the ordinary sense, $100_{10}$.)\n\nWhat is $b$?\n\nA new problem:\n\nIn a certain base $b$, the number of distinct digits used in all four-digit numbers is $b-1$. Find the sum of all such bases $b$.', 'Let $a$, $b$, and $c$ be positive real numbers such that $a + b + c = 1$. Find the maximum value of\n\\[\\frac{ab}{a + b} + \\frac{bc}{b + c} + \\frac{ca}{c + a}.\\]', 'A bookstore sells a total of 500 books per day, consisting of hardcover and paperback copies. The number of hardcover books sold daily is a multiple of 5, while the number of paperback books sold daily is a multiple of 3. If the average number of pages in a book sold daily is 250, and the average number of pages in a hardcover book is 10 more than the average number of pages in a paperback book, what is the ratio of the number of hardcover books sold to the number of paperback books sold?', "Consider the following functions: \n\nf(x) = x^3 + ax^2 - 2x + 1 \n\ng(x) = x^2 + bx - 3 \n\nDetermine the values of 'a' and 'b' such that (f(3) - g(2)) / (f(2) - g(3)) = 2."]}
_pin_dir = _os.environ.get('PRISM_CALIB_PIN_DIR', '/kaggle/working/calib_pins')
_pathlib.Path(_pin_dir).mkdir(parents=True, exist_ok=True)
_pathlib.Path(_pin_dir, 'math__Qwen_Qwen3.5_0.8B__n8__min64__max64__plain.json').write_text(
    _json.dumps(_PIN, ensure_ascii=False), encoding='utf-8')
_os.environ['PRISM_CALIB_PIN_DIR'] = _pin_dir
_os.environ['PRISM_CALIB_REQUIRE_PIN'] = '1'
print('[calib-pin] math: fp=' + _PIN['fp'] + ' n=' + str(len(_PIN['texts'])) + ' ключ=' + str(_PIN['protocol']))


In [ ]:
# ---------- pinned calibration pool: code ----------
import json as _json, os as _os, pathlib as _pathlib
_PIN = {'protocol': {'domain': 'code', 'model': 'Qwen/Qwen3.5-0.8B', 'n_samples': 8, 'min_length': 64, 'max_length': 64, 'offset_mapping': False}, 'source': {'dataset': 'neyralabs/apex-agents', 'config': None, 'split': 'train', 'field': 'prompt'}, 'fp': 'e02cdb39ed828cb2', 'texts': ["TAC and its supplier executed the first draft of the Master Supply Agreement document (Template 1)  as of April 1, 2025. President Trump announced additional tariffs on July 20, 2025. The supplier's raw materials come from Iraq. The supplier sends TAC notice that it intends to invoke the force majeure clause as a result of the tariffs. May the supplier do so under the terms of the agreement? Tell me your main findings by replying to me here.", 'TRIDENT AUTO CORPORATION (The "Plaintiff") has filed a claim challenging a 25% tariff that President Trump (the "Defendant") imposed on goods imported from China under the International Emergencies Economic Powers Act (IEEPA). The case is being heard by Judge Rudolph Contreras in the United States District Court for the District of Columbia.  Will the Plaintiff succeed in challenging the tariff? \n\nProvide me with a yes or no answer and a single sentence explanation. Reply straight back here. ', 'Review the two supply agreement templates, Master Supply Agreement Template 1.docx and Master Supply Agreement Template 4.pdf, along with the attached files (UCC §2-209 and Restatement (Second) of Contracts §89) to determine whether the supplier’s tariff-based request for a mid-term price increase would be a valid modification. Reply to me here with your assessment. ', 'TRIDENT AUTO CORPORATION (The "Plaintiff") has filed a Complaint against United States President Donald Trump (the "Defendant") in the United States District Court for the District of Columbia challenging a 30% tariff that the Defendant imposed on imports for metals that the Plaintiff uses in manufacturing. The Plaintiff has claimed that the International Emergency Economic Powers Act ("IEEPA") does not grant the Defendant to impose tariffs.  The Defendant has moved to transfer the action to the Court of International Trade. The case has been assigned to Judge Rudolph Contreras. \n\nWill the motion be granted? Give me a reply with a yes or no answer and a single sentence explanation. ', 'Due to a riot occurring in response to an Executive Order that resulted in the closure of its factories, TAC sought relief from performance under the force majeure section of the Master Supply Agreement. Citing the attached case, Buyer asserts that TAC is not excused from performance.\n\nIs Buyer correct? Provide your response in here with the following: "Yes/No"; and brief explanation.', 'Can you take a look at the two Master Supply Agreement templates (Master Supply Agreement Template.pdf ("Template1"), Master Supply Agreement 2.pdf ("Template2"))?  We’re considering them for Acme (the steel supplier) and we want a comparison.\n\nI need to know how each template deals with tariff‑related cost exposure, since Acme is importing steel from outside USMCA and the new tariffs are creating real financial pressure. Also, TAC is thinking about giving Acme a cash infusion secured by a lien on their receivables, but we’re worried about what happens if Acme goes bankrupt. \n\nCould you assess whether that financing structure would expose TAC to creditor claims, and which template gives TAC the most operational control? Please point to the clauses that support your analysis. Now, send me a clear text summary straight in here.', 'TRIDENT AUTO CORPORATION (The "Plaintiff") has filed a claim in the United States District Court for the District of Columbia against several federal government agencies (The "Defendants").  The Claim alleges that the Defendants have violated the Administrative Procedure Act (The "APA") by implementing several tariffs issued by Donald Trump under the International Emergencies Economic Powers Act ("IEEPA"). The claim is being heard in the United States District Court for the District of Columbia. The presiding Judge is Rudolph Contreras. \n\nWill the Plaintiffs succeed in their claim? Reply back to me with a Yes or No answer and a single sentence explanation. ', 'For the Kenvue deal, please send over the below draft figures for pre-deal target multiples for FY24. Utilize potential median deal value\n\nReturn to me a message with: Deal value/EBITDA, Deal value/EBIT, and Deal value/OpFCF. Round all values to one decimal place.']}
_pin_dir = _os.environ.get('PRISM_CALIB_PIN_DIR', '/kaggle/working/calib_pins')
_pathlib.Path(_pin_dir).mkdir(parents=True, exist_ok=True)
_pathlib.Path(_pin_dir, 'code__Qwen_Qwen3.5_0.8B__n8__min64__max64__plain.json').write_text(
    _json.dumps(_PIN, ensure_ascii=False), encoding='utf-8')
_os.environ['PRISM_CALIB_PIN_DIR'] = _pin_dir
_os.environ['PRISM_CALIB_REQUIRE_PIN'] = '1'
print('[calib-pin] code: fp=' + _PIN['fp'] + ' n=' + str(len(_PIN['texts'])) + ' ключ=' + str(_PIN['protocol']))


In [ ]:
# ---------- pinned calibration pool: chat ----------
import json as _json, os as _os, pathlib as _pathlib
_PIN = {'protocol': {'domain': 'chat', 'model': 'Qwen/Qwen3.5-0.8B', 'n_samples': 8, 'min_length': 64, 'max_length': 64, 'offset_mapping': False}, 'source': {'dataset': 'HuggingFaceH4/ultrachat_200k', 'config': None, 'split': 'train_sft', 'field': 'messages'}, 'fp': '4d3aab66d0e7aac6', 'texts': ["These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material? This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material. Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections? Sure, here are the steps to enable the secondary image hover feature on your Collection pages and Featured Collections sections:\n\n1. Log in to your Shopify account and go to your Online Store.\n2. Click on Customize theme for the section-based theme you are using.\n3. Navigate to the Collection pages or Featured Collections section where you want to enable the secondary image hover feature.\n4. With the section open, click on the section's setting (gear) icon in the top-left corner.\n5. In the settings panel that appears, look for an option labeled 'Image display' or 'Image hover'.\n6. If available, select 'Show secondary image on hover'.\n7. Save the changes and preview the Collection/Featured Collection page to see the effect.\n\nIf you're having trouble finding the setting, the best thing to do is refer to your theme's documentation, since the location and labeling of settings can vary between themes. Can you provide me with a link to the documentation for my theme? I don't have access to your store's theme information. However, you can usually find the documentation for your theme by going to the shopify theme store, finding your theme and clicking on the 'support' link located in the bottom right corner of the page. Alternatively, you can do a google search for the name of your theme followed by 'documentation' or 'user guide'. Can you confirm if this feature also works for the Quick Shop section of my theme? The secondary image hover feature may or may not work for your Quick Shop section, depending on the configuration of your theme. Some themes include this feature in the Quick Shop section by default, while others may require additional customization. To check if this feature is available for the Quick Shop section of your theme, follow these steps:\n\n1. Go to the Quick Shop section where you would like to enable the feature. 2. Click on the Quick Shop settings icon (gear icon) and look for 'Image display' or 'Image hover'. 3. If available, select 'Show secondary image on hover'. 4. Save the changes. If this option is not available in your Quick Shop section settings, you may need to reach out to your theme developer for assistance with customizing your Quick Shop section to include this feature.", 'Which famous landmarks should I visit in London, beyond the usual ones? 1. Leadenhall Market - a beautiful indoor market with stunning Victorian architecture, also used as a filming location in the Harry Potter films.\n\n2. St. Dunstan in the East - a ruined church in the middle of the city that has been turned into a beautiful public garden.\n\n3. The Monument - a 202-foot-tall column commemorating the Great Fire of London, with a staircase leading to a viewing platform offering great views of the city.\n\n4. The Camden Town Markets - an eclectic collection of markets offering food, fashion, and vintage items, plus live music and street performers.\n\n5. Novelist\'s House - the former home of Charles Dickens, now a museum dedicated to his life and works.\n\n6. The Old Operating Theatre - a museum housed in the oldest surviving operating theatre in Europe, with exhibits on the history of surgery and medical practices.\n\n7. The Churchill War Rooms - an underground complex used as a British government command centre during World War II, now a museum dedicated to Winston Churchill and his leadership during the war.\n\n8. The Brunel Museum - a museum dedicated to the life and works of engineer Isambard Kingdom Brunel, with exhibitions on his revolutionary designs for bridges, ships, and tunnels.\n\n9. The Clink Prison Museum - a museum in the former site of the notorious medieval prison, with exhibits on the history of crime and punishment in London.\n\n10. The Royal Observatory Greenwich - an observatory and museum dedicated to astronomy and navigation, with exhibits on the history of timekeeping and space exploration. Hmm, those are interesting suggestions, but I\'ve already seen so many touristy landmarks in London. Is there something more offbeat that you can recommend? Something that locals might know about? Absolutely! Here are some offbeat and lesser-known things to see in London that locals might recommend:\n\n1. God\'s Own Junkyard - a neon wonderland filled with vintage and new neon signs.\n\n2. Hornsey Town Hall Arts Centre - a grade II listed building with a rich history and home to many art exhibitions and events.\n\n3. Clapham Grand - a historic Victorian theatre that has been transformed into a music venue, cinema and a comedy club.\n\n4. Gordon Museum of Pathology - the pathology museum of King\'s College in London that holds a collection of over 8,000 medical specimens.\n\n5. John Soane\'s Museum - the former home of wealthy architect John Soane is a museum with eclectic collections of paintings, sculptures, and curiosities.\n\n6. The Viktor Wynd Museum of Curiosities - a small museum filled with strange and unusual objects, taxidermy, and rare books.\n\n7. The Cross Bones Graveyard - a medieval graveyard that became known as an outcast cemetery with a history connected to the sex industry, now a memorial garden.\n\n8. Highgate Cemetery - a beautiful Victorian graveyard known for its Gothic architecture and where many famous people, including Karl Marx, are buried.\n\n9. Dalston Eastern Curve Garden - a beautiful community garden near the Dalston Junction station that is run by volunteers and has regular events.\n\n10. The Geffrye Museum of the Home - a museum that explores the history of domestic interiors in England, with a beautiful garden and regular market events.\n\nThere are many other hidden gems in London, and a quick Google search for ‘offbeat things in London’ will bring up many blogs and resources with more options. Wow, these offbeat suggestions are really interesting. I had no idea there were so many unique things to see in London. Thanks for the recommendations, you\'re much more helpful than I expected. Can you tell me more about the Viktor Wynd Museum of Curiosities? It sounds like my kind of place. Sure! The Viktor Wynd Museum of Curiosities, Fine Art & Natural History is a small museum located in Hackney, East London. The museum, founded by collector Viktor Wynd is a place where you can find taxidermy, books, anatomical models, and other curiosities. The museum is famous for its collection of shrunken heads, dodo bones, and the famous mermaid. It is also notable for its dimly lit, atmospheric decor, and its bar which serves cocktails and allows visitors to take drinks into the museum. The museum is open daily except for Mondays and Tuesdays, and it also frequently hosts events and exhibitions, so be sure to check the museum\'s website to see what\'s on. Woah, the Viktor Wynd Museum sounds like a wacky and fun place to visit. I don\'t think I\'ve ever seen a shrunken head before. I wonder what kind of cocktails they serve there... Maybe I can convince some friends to check it out with me! Yes, the museum is definitely one of the more unusual places to visit in London! The cocktail menu at Viktor Wynd\'s bar is just as eclectic as the museum\'s collection, featuring brightly colored cocktails inspired by the exhibits, such as the "Shrunken Head" or "Mermaid Tail" cocktails. The museum and bar are on the smaller side, so it would make for a unique and intimate outing with friends. If you\'re looking for something different and memorable to do in London, the Viktor Wynd Museum is definitely worth checking out! Whoa, a bar in a museum?! That\'s wild. I\'m definitely going to have to try one of those cocktails. Do you think they have anything with a shrunken head as a garnish? That would be a trip. I am not sure if the Viktor Wynd Museum of Curiosities serves cocktails with an actual shrunken head as a garnish. However, they might have some creative cocktails that incorporate a shrunken head replica or something similar as an element in the drink. Nevertheless, the bar offers a unique setting and a cocktail menu that includes some unusual ingredients so it\'s definitely worth checking out! It\'s likely to be an eccentric and memorable experience, and I\'m sure you\'ll have a great time there with your friends. I can\'t wait to see that mermaid at the Viktor Wynd Museum. I wonder if it\'s real or just a replica. Either way, it\'s bound to be a fascinating item to behold. And I\'ll make sure to order that "Mermaid Tail" cocktail while I\'m there! Yes, the mermaid at the Viktor Wynd Museum of Curiosities is definitely one of its most famous exhibits. It\'s actually a recreation made from the skeleton of a monkey and the tail of a fish, so it\'s not a real mermaid of course, but it\'s still an intriguing piece. The museum has many other curiosities to discover as well, and the atmosphere there is unlike anything you\'ll find in other museums. And don\'t forget to try that "Mermaid Tail" cocktail - it\'s the perfect accompaniment to the museum\'s quirky vibe!', 'Write a comprehensive blog post of at least 1000 words about the top 10 most eco-friendly cities in the world and their renewable energy initiatives. Use a formal and informative tone, and include statistics, case studies, and expert opinions to support your claims. Make sure to cover various aspects of sustainability, such as public transportation, waste management, green buildings, and carbon emissions reduction. Additionally, provide specific examples of innovative renewable energy projects implemented in each city and assess their impact on the environment and the local community. Provide credible sources and links to relevant articles, reports, and websites to enhance the article\'s credibility and appeal to the readers. In the recent years, there has been a growing awareness of the necessity of developing sustainable cities to combat the detrimental effects of climate change. Cities are responsible for 75% of the world\'s carbon emissions, but they also have the potential to be leaders in renewable energy and green initiatives. Many cities are taking action to reduce their carbon footprint, and ten cities have gone above and beyond in terms of developing eco-friendly initiatives. In this post, we will take a look at the top ten most eco-friendly cities in the world and their renewable energy initiatives.\n\n1. Reykjavik, Iceland\nReykjavik is the capital of Iceland, and it is considered one of the most environmentally friendly cities in the world. The city is powered almost entirely by renewable energy, thanks to Iceland\'s abundant geothermal and hydroelectric power sources. About 90% of Reykjavik\'s population is connected to district heating, which is generated through geothermal power. The city also has an extensive public transportation system that relies on electric buses and taxis.\n\nOne of the most innovative renewable energy projects in Reykjavik is the CarbFix project, which involves capturing carbon dioxide emissions and storing them in underground basaltic rock formations. This initiative not only reduces carbon emissions, but it also creates a new type of renewable geothermal energy source. This project has reduced Reykjavik’s carbon footprint significantly and is gaining recognition worldwide.\n\n2. Copenhagen, Denmark\nCopenhagen is another city that has earned its reputation as a sustainable city. The city\'s ambitious goal is to be carbon-neutral by 2025, and it has already made significant progress in reducing its carbon emissions. Copenhagen has an extensive bicycle infrastructure, and biking is the most popular mode of transportation in the city. The city also has a robust public transportation system, with most buses running on biogas or electricity.\n\nCopenhagen is also home to several innovative energy-generating projects. The city’s Amager Bakke waste-to-energy plant is an excellent example of how to tackle waste management while generating clean energy. The plant incinerates waste, and the heat generated is converted into electricity, which is then supplied to the grid. The plant also features a recreational space with a ski slope, hiking trails, and even a climbing wall, making it more than just an energy-generating facility.\n\n3. Stockholm, Sweden\nStockholm is another city that has made significant progress in becoming more environmentally friendly. The city has a 100% renewable energy target by 2040, and it is well on its way towards achieving that goal. Like Copenhagen, Stockholm has an extensive bicycle infrastructure, with bikes being the most popular mode of transportation. The city also has an efficient public transportation system, which runs on renewable energy.\n\nStockholm has several innovative projects in the renewable energy sector. One such project is the Värtahamnen district cooling plant, which uses seawater to cool buildings. The cold water is pumped from deep-sea depth and is circulated through the city\'s cooling system. This system is significantly more energy-efficient than traditional air conditioning methods, and it has reduced Stockholm\'s carbon emissions significantly.\n\n4. Vancouver, Canada\nVancouver is a city known for its natural beauty, and it is also known for its commitment to sustainability. The city aims to be the greenest city in the world by 2020, and it has made significant progress towards that goal. The city has an impressive public transportation system, which includes buses, trains, and ferries, all of which run on renewable energy.\n\nVancouver has been investing in renewable energy initiatives and the green economy, creating new jobs while reducing carbon emissions. It is home to Canada’s largest biomass energy plant, which converts wood waste and sawdust into electricity, reducing the use of fossil fuels. Additionally, the city has launched an ambitious program called Renewable City Strategy, which aims to see the city run on 100% renewable energy before 2050.\n\n5. Portland, USA\nPortland is a city that has long been known for its eco-friendliness, and it has made significant strides in reducing its carbon emissions. The city aims to reduce its carbon emissions by 80% by 2050, and it has implemented several sustainable initiatives. Portland\'s public transportation system is extensive, and the city has invested heavily in bike lanes and bike-sharing programs, making cycling a popular option for getting around.\n\n Portland is also home to several innovative energy projects. One such project is the Covanta Portland waste-to-energy facility, which converts waste into clean energy. The project generates enough electricity to power more than 27,000 households and reduces the amount of waste sent to landfills significantly.\n\n6. San Francisco, USA\nSan Francisco is a city that has always been at the forefront of sustainability initiatives, and it aims to be a zero-waste city by 2020. The city has an impressive public transportation system, which includes buses, trains, and cable cars, all of which are powered by renewable energy sources. Additionally, San Francisco is home to the largest municipal solar installation in This is a really informative post about the top eco-friendly cities and their renewable energy initiatives. I would like to know more about their waste management strategies since it is also an important aspect of sustainability. Can you add some more information on that? Absolutely! Waste management is an important aspect of sustainability as it helps cities reduce their carbon footprint and ensure that resources are not wasted. Here is some information on the waste management strategies that these cities have implemented:\n\n7. Oslo, Norway\nOslo is the capital of Norway and is known for its impressive waste management program. The city is well on its way to becoming a zero-waste city, with over 60% of its waste being recycled or reused. Oslo\'s waste management program involves separating organic waste into biowaste, which is then converted into biogas and biofertilizer. Additionally, the city has implemented a "pay as you throw" policy, which has encouraged residents to reduce their waste output.\n\n8. Zurich, Switzerland\nZurich has a strong reputation for being environmentally friendly, and its waste management program is no exception. The city has a Zero Waste Strategy, which aims to have no waste sent to landfills by 2025. Zurich has an extensive recycling program, and residents are encouraged to separate their waste into different categories, including paper, glass, metal, and plastic. Organic waste is also collected separately and composted, which is then used to fertilize public parks and gardens.\n\n9. Adelaide, Australia\nAdelaide is a city that has made significant progress in reducing its carbon emissions, and its waste management program is contributing to that goal. The city has implemented a system where household waste is sorted into three different bins: one for organic waste, one for recyclable materials, and one for general waste. The organic waste is processed into compost, which is then used to fertilize public parks and gardens, while recyclable materials are sent to recycling centers.\n\n10. Singapore\nSingapore is a city-state that has made waves with its innovative waste management program. The city is known for its high population density, which makes waste management a significant challenge. Singapore has implemented a "3R" approach - reduce, reuse, and recycle - to reduce its waste output. The city also has a waste-to-energy plant, which converts waste into electricity that is then supplied to the grid.\n\nIn conclusion, all the top ten most eco-friendly cities in the world have implemented various strategies for waste management. By focusing on sustainable waste management strategies, these cities have been able to reduce their carbon emissions, protect the environment, and promote sustainability. Thank you for the additional information about waste management strategies in these eco-friendly cities. I was also wondering if you could provide more details on their green building initiatives, such as the use of sustainable materials and designs to reduce energy consumption and carbon emissions. Certainly! Green buildings are an important aspect of sustainability, as they can significantly reduce energy consumption and help lower carbon emissions. Here are some examples of green building initiatives in these eco-friendly cities:\n\n1. Reykjavik, Iceland\nReykjavik has implemented several programs to promote sustainable building practices. The city has a "Green Steps" program, which encourages residents and businesses to adopt sustainable practices. Reykjavik has also mandated that all new buildings must meet strict energy efficiency standards. Buildings are designed to maximize natural light and use sustainable building materials wherever possible.\n\n2. Copenhagen, Denmark\nCopenhagen has implemented several green building initiatives, including the use of sustainable materials and designs to reduce energy consumption. The city has a program called "Green Roofs and Walls," which promotes the use of vegetation on buildings to improve energy efficiency and reduce the urban heat island effect. Additionally, Copenhagen has implemented a district heating program, which uses waste heat from power plants to heat buildings, reducing energy consumption.\n\n3. Stockholm, Sweden\nStockholm has implemented several programs to promote green building practices, including the use of sustainable materials and designs. The city has adopted strict energy efficiency standards for new buildings, and it promotes the use of green roofs and walls to improve energy efficiency and reduce the urban heat island effect. Stockholm also encourages the use of solar panels and other renewable energy sources in buildings.\n\n4. Vancouver, Canada\nVancouver has implemented several programs to promote sustainable building practices, including the use of sustainable materials and designs. The city has adopted strict green building codes, and it encourages the use of energy-efficient windows, insulation, and lighting. Vancouver also has a program that encourages the use of green roofs and walls to improve energy efficiency and reduce the urban heat island effect.\n\n5. Portland, USA\nPortland is known for its sustainable building practices, and the city has implemented several green building initiatives. The city has implemented a green building policy, which encourages the use of sustainable materials and designs to reduce energy consumption. Additionally, Portland has a program called "Ecoroofs," which promotes the use of vegetation on roofs to improve energy efficiency and reduce the urban heat island effect.\n\n6. San Francisco, USA\nSan Francisco has implemented several programs to promote green building practices, including the use of sustainable materials and designs. The city has adopted strict energy efficiency standards for new buildings, and it promotes the use of green roofs and walls to improve energy efficiency and reduce the urban heat island effect. Additionally, San Francisco has a program called Zero Net Energy Buildings, which encourages the construction of buildings that generate as much energy as they use.\n\nIn conclusion, all the top ten most eco-friendly cities in the world have implemented programs to promote sustainable building practices. By using sustainable materials and designs, these cities have significantly reduced energy consumption and lowered their carbon emissions. I\'m impressed by the efforts these eco-friendly cities have made to promote sustainability. However, I\'m curious about the challenges these cities might have faced during the implementation of their renewable energy and sustainability initiatives. Can you provide me with more details on those challenges and how the cities overcame them? Certainly! While these ecologically friendly cities have been successful in implementing renewable energy and sustainability initiatives, they have also faced their fair share of challenges. Here are some examples of challenges faced by these cities and how they overcame them:\n\n1. Reykjavik, Iceland\nDespite its extensive use of renewable energy, Reykjavik faced the challenge of balancing its energy-intensive industries with sustainability goals. The city overcame this challenge by implementing a carbon capture project, keeping pace with the emission-intensive industries, while still reducing the carbon footprint. The CarbFix project, which I described earlier, proved to be a game-changer, making Reykjavik a city that effectively captured and utilized carbon dioxide. 2. Copenhagen, Denmark\nCopenhagen faced the challenge of encouraging car drivers to adopt more eco-friendly modes of transportation. The city overcame this challenge by incentivizing bike commuting, providing public transportations, and by raising awareness on the positive impacts of sustainable travel. The city was able to increase the number of bike riders to account for approximately 50% of commuters, contributing to the significant reduction of carbon emissions. 3. Stockholm, Sweden\nStockholm faced the challenge of finding a way to sustain green infrastructure projects despite the cost, particularly in a city where both land and property values are relatively high. The city overcame this challenge by cooperating with businesses and local community organizations to help defray the costs of infrastructure projects. By doing so, Stockholm effectively secured long-term funding for infrastructure projects that are vital for sustainability. 4. Vancouver, Canada\nVancouver faced the challenge of reducing carbon emissions while still maintaining economic growth. The city overcame this challenge by promoting the development of sustainable industries, while at the same time fostering an eco-friendly environment for businesses. By doing so, local businesses were encouraged to adopt more sustainable practices and were recognized for their efforts in promoting sustainability. 5. Portland, USA\nPortland faced the challenge of dealing with a lack of federal support for sustainability initiatives, particularly during the early stages of their implementation. The city overcame this challenge by forging partnerships with local businesses and organizations to promote sustainability initiatives. By embracing public-private partnerships and working closely with the city\'s commercial sector, Portland was able to overcome a lack of federal support while still promoting sustainable practices. 6. San Francisco, USA\nSan Francisco faced the challenge of competing for resources with other nearby communities, given its geographically small size. The city overcame this challenge by adopting a collaborative approach with other nearby communities, encouraging them to adopt eco-friendly practices. By doing this, San Francisco was able to leverage the resources of neighboring communities to promote sustainability initiatives throughout the region. In conclusion, while these eco-friendly cities have faced their share of challenges in promoting sustainability, they have successfully overcome them. Through collaborating with businesses and community organizations, promoting sustainable practices, and embracing partnerships, these cities have successfully implemented different renewable energy and green initiatives, promoting a more sustainable future.', "De León, previewing the speech he will give today, said he will highlight his Senate Bill 535, which directs a quarter of the proceeds from the Greenhouse Gas Reduction Fund to projects that benefit disadvantaged communities.\nOn Thursday, de León nodded approvingly as a string of leading scientists and religious leaders gathered for hours of weedy policy discussions on the impacts of climate change, including gloomy predictions on mortality attributable to air pollution.\nSEIU HEADS TO THE BAR: Employees of the State Bar of California represented by SEIU are planning a picket line today at the bar building in Los Angeles to protest the latest contract offer. What is the reason for SEIU employees planning a picket line at the State Bar of California building in Los Angeles? The reason for SEIU employees planning a picket line at the State Bar of California building in Los Angeles is to protest the latest contract offer. Can you provide more details about Senate Bill 535 and how it will benefit disadvantaged communities? Sure, Senate Bill 535 is a California law that directs at least 25% of the money generated through the state's Greenhouse Gas Reduction Fund towards projects that benefit disadvantaged communities. The bill is designed to ensure that the benefits of California's climate policies, which include cap-and-trade, are felt most heavily by communities that have been disproportionately impacted by pollution and environmental degradation.\n\nThe funds generated through the bill are invested in projects that promote clean energy, reduce pollution, and support sustainable communities. This includes initiatives like public transportation, energy efficiency retrofits for low-income homes, and urban forestry projects.\n\nProponents of the bill argue that it helps to level the playing field for low-income communities and communities of color, which have historically faced greater exposure to environmental hazards and pollution. By directing funds towards these communities, the bill aims to create more equitable and sustainable outcomes for all Californians.", 'Write an essay that evaluates the positive and negative influence of social media on personal relationships, citing specific examples and research to support your claims. Analyze the ways in which social media usage affects communication patterns, trust levels, and emotional connections in romantic partnerships, friendships, and family relationships. Consider the role of social comparison, cyberbullying, and privacy concerns in shaping the impact of social media use on personal relationships. Finally, provide recommendations for how individuals can cultivate healthy and meaningful relationships in the age of social media. Social media is a ubiquitous presence in modern society and has revolutionized the way people communicate with one another. While there are undoubtedly positive aspects of social media usage, there are also significant negative consequences that impact personal relationships. In this essay, I will evaluate the positive and negative influence of social media on personal relationships, citing specific examples and research to support my claims. I will analyze the ways in which social media usage affects communication patterns, trust levels, and emotional connections in romantic partnerships, friendships, and family relationships. I will also consider the role that social comparison, cyberbullying, and privacy concerns play in shaping the impact of social media use on personal relationships. Finally, I will provide recommendations for how individuals can cultivate healthy and meaningful relationships in the age of social media.\n\nOne of the positive aspects of social media is that it allows people to communicate with one another more easily and frequently than ever before. Individuals can maintain long-distance relationships with friends and family members, interact with people who share similar interests or backgrounds, and cultivate new friendships with people from all over the world. Social media platforms like Facebook and Instagram provide an easy way for users to share their lives with others, which can enhance emotional connections and feelings of intimacy in relationships. For example, a recent study found that social media use was positively associated with self-disclosure, which is a key factor in building and maintaining close relationships (Chen & Lee, 2013).\n\nHowever, social media also has negative consequences that can impact personal relationships. One of the most significant concerns is the impact of social comparison, which can lead to feelings of envy, insecurity, and dissatisfaction. Studies have shown that frequent social media use is associated with higher levels of depressive symptoms, and that exposure to idealized and curated images of others can negatively impact self-esteem (Vogel et al., 2014). Additionally, social media can provide a platform for cyberbullying, which can cause emotional distress, anxiety, and even suicide. Cyberbullying can come from strangers, but it can also come from people who are known to the victim, such as peers or romantic partners. A recent survey found that one in four teenagers has experienced cyberbullying, and that it can lead to decreased self-esteem and social isolation (Hinduja & Patchin, 2018).\n\nMoreover, the overreliance on social media for communication can also have negative effects on personal relationships. In romantic partnerships, excessive social media use has been associated with lower levels of trust and relationship satisfaction (Muise et al., 2009). Social media usage can lead to misinterpretation of messages, and the lack of nonverbal cues can make it difficult to understand the true meaning of a conversation. In families, the overuse of social media can hinder face-to-face communication and physical interaction, which are essential components of healthy relationships.\n\nBased on these findings, individuals should take steps to limit the negative impact of social media on their personal relationships. One way to do so is to cultivate meaningful offline interactions with friends and family members. Taking a break from social media can also be beneficial, particularly if it leads to more in-person communication and interaction. Additionally, individuals can be mindful of the content they post on social media, and strive to create realistic and authentic portrayals of their lives. Finally, it is essential to educate oneself on the dangers of cyberbullying and to take active steps to prevent it.\n\nIn conclusion, social media has both positive and negative influences on personal relationships. While social media can enhance communication and provide a platform for building new relationships, it can also result in social comparison, cyberbullying, and a lack of trust in personal relationships. By taking steps to limit the negative effects of social media exposure and cultivating healthy, in-person relationships, individuals can create meaningful and fulfilling connections with others while still enjoying the benefits of social media use. Can you add more information on the impact of social media on family relationships? Also, can you suggest some practical tips on how to balance social media use and face-to-face communication with our loved ones? Sure, here are some additional details on the impact of social media on family relationships. Social media has transformed the way we interact with our families, with both positive and negative implications. While it can provide a platform to stay in touch with family members living far away or separated by distance, it can also hinder close relationships by replacing face-to-face communication with online interactions. Social media has been found to decrease the amount of quality time families spend together physically, which can affect the emotional connections between family members. For example, studies have found that children who spend more time on social media have lower relationship satisfaction with their parents, while parents who spend more time on social media have lower relationship satisfaction with their children (Kross et al., 2020). Moreover, the content shared on social media can sometimes lead to misunderstandings and conflicts in family relationships. Parents may share pictures or posts related to their child without their consent, which can result in embarrassment or frustration. Similarly, siblings may share information about their siblings without their permission, which can strain relationships between siblings. That being said, there are practical tips to balance social media use and face-to-face communication with our loved ones. The first step is to set boundaries and establish times when social media will be avoided or limited. For example, putting aside an hour or two a day to disconnect from social media and spend time with family can make a big difference in fostering healthy relationships. Family members can also establish a "no-phone" policy during meal times or during important family events such as holidays or vacations. Seeking out opportunities to engage in shared activities can also strengthen family bonds. Activities such as cooking, hiking, or game nights can create opportunities for meaningful interactions and laughter, which can foster stronger emotional connections between family members. Finally, finding ways to use social media to enhance family relationships is important. Tools such as video chat or group messaging platforms can allow family members to stay in touch and share their lives with one another while also maintaining meaningful connections. Overall, balancing social media use with face-to-face communication and shared activities can help individuals cultivate healthy and meaningful family relationships while still enjoying the benefits of social media use. Can you give some examples of "shared activities" that families can do together to strengthen their relationships? Sure! Here are a few examples of shared activities that families can do together to strengthen their relationships:\n\n1. Cooking and baking: Preparing meals together is not only a practical activity that can help families save time and money, but it also provides an opportunity for sharing stories, tips, and traditions. Cooking and baking can be a fun way for family members to connect and create memories while enjoying tasty treats. 2. Outdoor activities: Hiking, cycling, or camping are all great ways to get outdoors and enjoy nature. Family members can take turns planning and organizing trips or activities, and encourage each other to try new things. Outdoor activities also provide an opportunity to disconnect from technology and spend quality time together. 3. Game nights: Board games, card games, or video games can be a fun way for family members to engage in friendly competition and create shared experiences. Game nights can be an opportunity to learn new games or revisit old favorites, and can be enjoyed by family members of all ages. 4. Movie nights: Watching movies together allows family members to bond over shared interests and experiences. Family members can take turns selecting movies, and share their thoughts and opinions during or after the screening. Movie nights can also be an opportunity to introduce younger family members to classic or new movies that they may not have seen before. 5. Volunteer work: Participating in volunteer work as a family can provide a sense of purpose and fulfillment, while also strengthening relationships. Family members can choose a cause or organization that they feel passionate about, and work together towards a common goal. Volunteering is also a great way to instill values of empathy and compassion in younger family members. Overall, shared activities should be fun, engaging, and flexible enough to accommodate everyone\'s interests and schedules. By finding ways to spend time together as a family, and creating meaningful connections, individuals can create a positive and healthy family environment. Can you give some more examples of outdoor activities that families can enjoy together? Sure, here are some more examples of outdoor activities that families can enjoy together:\n\n1. Picnics: Going on a picnic is a great way to spend time together outdoors while enjoying a meal or snack. Families can pack a basket of their favorite foods and drinks, and find a scenic location in nature, such as a park, beach, or forest. 2. Geocaching: Geocaching is an outdoor treasure hunt that uses GPS coordinates to locate hidden "caches" containing small treasures or messages. Families can work together to discover new caches and share their finds with one another. 3. Nature walks: Taking a leisurely stroll in nature can be a great way to relax and bond as a family. Families can choose a trail or park to explore, and spend time observing and discussing the plants, animals, and natural features they encounter along the way. 4. Water activities: If a family lives near a lake, river, or ocean, they can enjoy activities such as swimming, kayaking, or paddleboarding. These activities provide a fun and exciting challenge that the whole family can enjoy together. 5. Gardening: Gardening is a great way to connect with nature and engage in a productive, creative activity. Families can work together to plan, plant, and care for a garden, and enjoy the fruits of their labor as a family. Overall, outdoor activities provide an excellent opportunity for families to spend quality time together while enjoying the benefits of fresh air, exercise, and nature. Families can choose activities that suit their interests, skills, and comfort levels, and use them as a way to build strong and lasting relationships.', "What many companies do when planning their global marketing strategies is to organize the world’s countries into three categories.\nEmerging Markets – Markets regarded as having the most potential for growth (e.g. Huge countries such as China and India) and targeted for long-term investment.\nRest of the World (ROW) Markets – Markets that are often not included in a global marketing strategy because a company does not have sufficient time or resources to pay them any attention.\nA typical company involved in global marketing is likely to have four or five core markets and maybe six to twelve emerging markets.\nIs it Wise to Ignore the ROW?\nCollectively the ROW consists of a huge market of over a billion people. Of course the ROW includes a lot of small countries, but even a country with only five to six million people (e.g. Denmark and Ireland) could be very profitable for the right product. Click here for a list of countries in order of population.\nWhich Languages Does the “Rest of the World” Speak?\nThe Web Globalization Report Card, which tracks the websites of 150 global companies, reports a steadily increasing number of languages supported by global content. As of 2018, the average number of languages supported by all 150 global websites is now thirty-two.\nThe success of global marketing can only be achieved through the use of local languages. If the growth in the number of languages is imagined as a curve, this curve begins quite steeply as companies load up on the core European and Asian languages. After a company reaches the level of twenty languages, the curve starts to taper off. The number of languages supported by a few companies, such as Facebook, Visa, and Dyson has shot up in sudden spikes. However, most companies are only in the early stages of this growth curve.\nIt would be a herculean task to utilize the more than 6,000 languages spoken worldwide, so a global marketing company has to narrow down its language choices to the languages spoken by its core target audiences. The following is a list of ten languages that are shared by leading global websites.\nEnglish – English is the language of globalization and is spoken by over 75% of the world’s population.\nChinese (Simplified) – If people in Chinese speaking markets continue to expand their internet usage, Chinese might one day supplant English as the most widely used internet language.\nArabic – The Arabic language is spoken by 295 million speakers worldwide, and is the official language of 28 different countries. A report from the British Council ranks Arabic as the second most important language of the future.\nSpanish – There are a number of important Spanish speaking countries, not to mention that the Hispanic population in the US will likely double by 2050.\nGerman – The German language is the fourth most spoken language in the world with nearly 210 million speakers worldwide.\nPortuguese – Second behind Spanish in Latin America. Portuguese is spoken by 215 million people in Portugal, Brazil, and some parts of Africa.\nFrench – French is the official language of over 29 countries throughout the world including several of the fastest growing African economies.\nJapanese – Japanese has 130 million native speakers and a similar number of non-native speakers spread out over the world.\nRussian – There are 160 million native speakers in Russia and throughout central and Eastern Europe.\nHindi – Although India is home to 126 million English speakers, around 85% of the population doesn’t speak English well or at all. India represents an enormous growing market.\nIf you are hoping to increase your global marketing reach, it’s essential to expand the number of languages your website or app supports. All countries, including the “rest of the world,” are within your reach. Once you have decided on your target markets, Localize will help you translate your content into the local languages. Talk to us for more information.\nCan you provide a list of the top ten languages that global marketing companies should focus on, according to the leading global websites? Yes, here is a list of the top ten languages that global marketing companies should focus on, according to the leading global websites:\n\n1. English\n2. Chinese (Simplified)\n3. Arabic\n4. Spanish\n5. German\n6. Portuguese\n7. French\n8. Japanese\n9. Russian\n10. Hindi Can you please provide me with more details on how companies can effectively target emerging markets? Certainly! Here are some tips on how companies can effectively target emerging markets:\n\n1. Conduct Market Research: Before entering an emerging market, do your research. Understand the local culture, consumer preferences, and market trends. This information will help you tailor your products and marketing strategies to suit the needs of the local market. 2. Localize Your Marketing: Use local languages, cultural references, and imagery that resonates with the target audience. Avoid direct translations that may not make sense in the local context. 3. Build Local Partnerships: Partner with local businesses, distributors or agents who have industry know-how and established relationships in the local market. This can help overcome regulatory and distribution challenges. 4. Adapt Your Products: Products that are successful in established markets may not be relevant in emerging markets. Conduct market research to understand the unique needs of the local market, and adapt your products or services accordingly. 5. Price Sensitivity: Emerging markets are often price-sensitive, so it's essential to develop a pricing strategy that reflects the local market's purchasing power. 6. Build Trust: Emerging markets often lack confidence in international brands, especially if they come from developed countries. Build trust by investing in the local economy, recruiting local talent, and partnering with local suppliers. 7. Be Patient: Emerging markets require a long-term perspective. It takes time to establish a reputation, build relationships, and gain market share. Companies need to be patient and invest in the market for the long term. By following these tips, companies can effectively target emerging markets and tap into their potential for growth.", 'Write a Python program that can identify and remove duplicate images from a given folder or directory. The program should be able to compare images based on their content and file size, and determine which ones are exact duplicates. The duplicates should then be deleted or moved to a separate folder. The program should also provide a user-friendly interface with clear instructions on how to use it. I don\'t have access to the ui. However, I can provide you with code for the backend logics of the required program.\n\nhere is a sample code using the pillow library to identify and remove duplicate images:\n\n```\nfrom pil import image\nimport os\nimport hashlib\n\ndef get_size(img_path):\n    """calculate and return file size of image"""\n    size = os.path.getsize(img_path)\n    return size\n\ndef get_hash(img_path):\n    """calculate and return hash value of image"""\n    with open(img_path, \'rb\') as f:\n        img_hash = hashlib.md5(f.read()).hexdigest()\n    return img_hash\n\ndef find_duplicates(path):\n    """find duplicate images in a directory"""\n    hashes = {}\n    duplicates = {}\n    for root, dirs, files in os.walk(path):\n        for file in files:\n            img_path = os.path.join(root, file)\n            size = get_size(img_path)\n            img_hash = get_hash(img_path)\n            if img_hash in hashes.keys() and size == hashes[img_hash]:\n                if img_hash in duplicates.keys():\n                    duplicates[img_hash].append(img_path)\n                else:\n                    duplicates[img_hash] = [hashes[img_hash], img_path]\n            else:\n                hashes[img_hash] = size\n    return duplicates\n\ndef remove_duplicates(duplicates):\n    """remove duplicate images"""\n    for img1, img2 in duplicates.items():\n        for I in img2:\n            os.remove(i)\n```\n\nthe above code has two functions, `find_duplicates()` and `remove_duplicates()`. The former function takes a path argument and finds all the duplicate images in that directory. The latter function takes the dictionary of duplicate images returned by the `find_duplicates()` function and deletes the duplicate images.\n\nyou can use the above function call to identify duplicate files.\n\n```\nduplicates = find_duplicates(\'/path/to/directory\')\nremove_duplicates(duplicates)\n```\n\nnote: the above code won\'t remove the original image. It will remove all the other duplicates from a folder. Could you please add some comments to explain what each function does and how they work? It would be really helpful. Sure! Here\'s the code with comments:\n\n```\nfrom PIL import Image\nimport os\nimport hashlib\n\ndef get_size(img_path):\n    """Calculate and return file size of image"""\n    size = os.path.getsize(img_path) # Get size of image\n    return size\n\ndef get_hash(img_path):\n    """Calculate and return hash value of image"""\n    with open(img_path, \'rb\') as f:\n        img_hash = hashlib.md5(f.read()).hexdigest() # Calculate hash value of image\n    return img_hash\n\ndef find_duplicates(path):\n    """Find duplicate images in a directory"""\n    hashes = {} # Dictionary to store hash values and image sizes\n    duplicates = {} # Dictionary to store duplicate images\n    for root, dirs, files in os.walk(path): # Iterate over all files in directory\n        for file in files:\n            img_path = os.path.join(root, file) # Get image path\n            size = get_size(img_path)\n            img_hash = get_hash(img_path)\n            if img_hash in hashes.keys() and size == hashes[img_hash]: # If size and hash value match, image is duplicate. If img_hash in duplicates.keys(): # If dictionary already contains an entry for this hash value\n                    duplicates[img_hash].append(img_path) # Append the path of the duplicate to the list of duplicates for this hash\n                else:\n                    duplicates[img_hash] = [hashes[img_hash], img_path] # If this is the first duplicate for this hash value, add an entry to the dictionary with this hash and the path of the first occurrence of this hash\n            else: # If size and hash value don\'t match, image is not a duplicate\n                hashes[img_hash] = size # Add hash and size entry for new image\n    return duplicates # Return dictionary of duplicate images\n\ndef remove_duplicates(duplicates):\n    """Remove duplicate images"""\n    for img1, img2 in duplicates.items(): # Iterate over dictionary of duplicates\n        for I in img2:\n            os.remove(i) # Remove duplicate image\n```\n\nHope this helps! Let me know if you have any other questions. Can you please add some instructions or prompts for the user to enter the directory path? It would be great to have a user-friendly interface for this program. Sure! Here\'s a modified version of the code with added prompts for user input:\n\n```\nfrom PIL import Image\nimport os\nimport hashlib\n\ndef get_size(img_path):\n    """Calculate and return file size of image"""\n    size = os.path.getsize(img_path)\n    return size\n\ndef get_hash(img_path):\n    """Calculate and return hash value of image"""\n    with open(img_path, \'rb\') as f:\n        img_hash = hashlib.md5(f.read()).hexdigest()\n    return img_hash\n\n# Prompt user to enter directory path\ndir_path = input("Enter the path of the directory to check for duplicates: ")\n\ndef find_duplicates(path):\n    """Find duplicate images in a directory"""\n    hashes = {}\n    duplicates = {}\n    for root, dirs, files in os.walk(path):\n        for file in files:\n            img_path = os.path.join(root, file)\n            size = get_size(img_path)\n            img_hash = get_hash(img_path)\n            if img_hash in hashes.keys() and size == hashes[img_hash]:\n                if img_hash in duplicates.keys():\n                    duplicates[img_hash].append(img_path)\n                else:\n                    duplicates[img_hash] = [hashes[img_hash], img_path]\n            else:\n                hashes[img_hash] = size\n    return duplicates\n\ndef remove_duplicates(duplicates):\n    """Remove duplicate images"""\n    for img1, img2 in duplicates.items():\n        for I in img2:\n            os.remove(i)\n\nduplicates = find_duplicates(dir_path)\nremove_duplicates(duplicates)\n```\n\nNow, when the user runs the program, they will be prompted to enter the path of the directory they want to check for duplicates. The program will then use this path in the `find_duplicates()` function to identify and remove duplicate images. I hope this helps! Let me know if you have any other questions.', "Please provide a specific cooking method for the salmon, as well as any seasonings or ingredients necessary to complete the dish. Additionally, include instructions for how to properly prepare the salmon before cooking, such as removing the skin or deboning. Please write in step-by-step format, using clear and concise language. Pan-Seared Salmon with Lemon Butter Sauce\n\nIngredients:\n\n- 4 salmon fillets\n- Salt and pepper\n- 2 tablespoons olive oil\n- 1/4 cup unsalted butter\n- 2 cloves garlic, minced\n- 1/4 cup chicken or vegetable broth\n- 2 tablespoons lemon juice\n- Lemon wedges and chopped parsley, for garnish\n\nInstructions:\n\n1. Preheat the oven to 375°F (190°C).\n\n2. Season both sides of the salmon fillets with salt and pepper.\n\n3. If the salmon has skin, you may remove it by using a sharp knife to separate the skin from the flesh at one end. Then, grab the skin with your fingers and gently peel it off. You can also ask your fishmonger to skin the salmon for you.\n\n4. Heat the olive oil in a large oven-safe skillet over medium-high heat.\n\n5. Once the oil is hot, add the salmon fillets skin-side up or down, depending on whether or not you removed the skin. Cook for 3-4 minutes or until the bottom is golden brown and crispy.\n\n6. Use a spatula to carefully flip the salmon over, and transfer the skillet to the preheated oven. Bake the salmon for 5-7 minutes or until it is cooked through and flakes easily with a fork.\n\n7. While the salmon is baking, prepare the lemon butter sauce. Melt the butter in a small saucepan over medium heat.\n\n8. Add the minced garlic and cook for 1 minute or until fragrant.\n\n9. Pour in the chicken or vegetable broth and lemon juice. Bring the mixture to a simmer, and let it cook for 2-3 minutes or until slightly thickened.\n\n10. Taste the sauce and adjust the seasoning with salt and pepper as needed.\n\n11. Remove the skillet from the oven and transfer the salmon fillets to a serving platter.\n\n12. Drizzle the lemon butter sauce over the salmon, and garnish with lemon wedges and chopped parsley.\n\n13. Serve immediately, and enjoy your delicious and healthy meal! Could you please suggest some side dishes that would pair well with the pan-seared salmon with lemon butter sauce recipe? also, do you have any recommendations for a wine pairing? Sure, I'd be happy to suggest some side dishes and wine pairings for the pan-seared salmon with lemon butter sauce recipe! Side dishes:\n- Roasted vegetables (such as asparagus, Brussels sprouts, or broccoli)\n- Wild rice pilaf\n- Grilled or roasted potatoes\n- Quinoa or couscous salad with vegetables and herbs\n- Lemon garlic roasted cauliflower\n- Green beans with almonds and shallots\n\nWine pairings:\n- Chardonnay: It's a classic choice to pair with salmon, and the buttery flavor of the lemon sauce would complement the creamy texture of the wine. - Pinot Noir: Its light and fruity flavor pairs well with the delicate yet rich flavor of the salmon. - Sauvignon Blanc: Its bright acidity and citrus flavors would complement the lemon butter sauce well. - Rosé: A dry rosé would be a refreshing choice that would complement the flavors of both the salmon and the lemon butter sauce. I hope this helps, and enjoy your meal! Can you recommend some herb or spice blends that would pair nicely with the salmon dish? Also, do you have any recommendations for a dessert to finish off the meal? Sure, I'd be happy to recommend some herb or spice blends that would pair well with the pan-seared salmon with lemon butter sauce dish! Here are a few ideas:\n\n- Dill: This herb has a slightly sweet and tangy flavor that pairs really well with salmon. You could sprinkle some chopped fresh dill over the salmon just before serving, or use a dill seasoning blend. - Lemon pepper: This classic seasoning blend consists of cracked black pepper and dried lemon zest. It adds a nice citrusy kick to the salmon that would complement the lemon butter sauce. - Paprika: This mild and slightly sweet spice would add a smoky and slightly earthy flavor to the salmon. You could use a smoked paprika to enhance the smoky flavor. - Italian seasoning: This blend typically consists of dried basil, oregano, thyme, rosemary, and marjoram. It would add a nice herbal flavor to the salmon that would go well with the lemon butter sauce. As for dessert, here are a few ideas to finish off your meal:\n\n- Lemon tart: This would be a perfect way to carry on the citrusy theme of the meal. A buttery crust filled with tangy lemon curd would be the perfect way to end the meal. - Mixed berry shortcake: After a rich and savory meal, a light and fluffy dessert like shortcake topped with fresh mixed berries and whipped cream would be a refreshing way to end the meal. - Chocolate mousse: For chocolate lovers, a rich and decadent chocolate mousse would be the perfect indulgence after a healthy and light main course. I hope this gives you some ideas, and enjoy your meal and dessert!"]}
_pin_dir = _os.environ.get('PRISM_CALIB_PIN_DIR', '/kaggle/working/calib_pins')
_pathlib.Path(_pin_dir).mkdir(parents=True, exist_ok=True)
_pathlib.Path(_pin_dir, 'chat__Qwen_Qwen3.5_0.8B__n8__min64__max64__plain.json').write_text(
    _json.dumps(_PIN, ensure_ascii=False), encoding='utf-8')
_os.environ['PRISM_CALIB_PIN_DIR'] = _pin_dir
_os.environ['PRISM_CALIB_REQUIRE_PIN'] = '1'
print('[calib-pin] chat: fp=' + _PIN['fp'] + ' n=' + str(len(_PIN['texts'])) + ' ключ=' + str(_PIN['protocol']))


In [ ]:
import hashlib
import json
import os
import sys
import numpy as np
import torch
MIN_BIN_PAIRS = 5
N_BINS = 14
def _cos(u: torch.Tensor, v: torch.Tensor) -> float:
    num = (u * v).sum()
    den = u.pow(2).sum().sqrt() * v.pow(2).sum().sqrt()
    if den.item() < 1e-12:
        return float("nan")
    return (num / den).item()
def coh_curve(
    cos_deltas: list[tuple[float, float]], n_bins: int = N_BINS
) -> tuple[list[float], list[float]]:
    """Log-delta binning: (bin_centers, mean_cos). Drops bins with < MIN_BIN_PAIRS."""
    pts = [(d, c) for d, c in cos_deltas if d > 0 and c == c]
    if not pts:
        return [], []
    ds = np.array([p[0] for p in pts])
    # ponytail: edges padded by 2% so sigma_d slightly outside the pair range
    # still interpolates (v3: L6 sigma_d=1.663 vs min bin 1.67 -> None).
    edges = np.logspace(
        float(np.log10(ds.min())) - 0.01, float(np.log10(ds.max())) + 0.01, n_bins + 1
    )
    centers, means = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = [c for d, c in pts if lo <= d < hi]
        if len(sel) >= MIN_BIN_PAIRS:
            centers.append(float(np.sqrt(lo * hi)))
            means.append(float(np.mean(sel)))
    return centers, means
def ell_of(centers: list[float], means: list[float], level: float = 0.5) -> float | None:
    """delta where C crosses `level` (linear interp in log-delta). None = no crossing."""
    for i in range(len(means) - 1):
        a, b = means[i] - level, means[i + 1] - level
        if a * b <= 0 and a != b:
            t = a / (a - b)
            d0, d1 = np.log(centers[i]), np.log(centers[i + 1])
            return float(np.exp(d0 + t * (d1 - d0)))
    return None
def _pair_stats(js: list[torch.Tensor], hs: list[torch.Tensor]):
    """All C(n,2) pairs -> (cos_deltas, L_values)."""
    cos_deltas, ls = [], []
    n = len(js)
    for i in range(n):
        for j in range(i + 1, n):
            d = (hs[i] - hs[j]).norm().item()
            cos_deltas.append((d, _cos(js[i], js[j])))
            ls.append((js[i] - js[j]).norm().item() / max(d, 1e-12))
    return cos_deltas, ls
def _curve_from_pairs(
    cos_deltas: list[tuple[float, float]], ls: list[float], n_points: int
) -> dict:
    """C(delta) curve + ell + L stats from an EXACT list of pairs."""
    centers, means = coh_curve(cos_deltas)
    return {
        "n_points": n_points,
        "n_pairs": len(cos_deltas),
        "delta_centers": centers,
        "C": means,
        "ell_0.5": ell_of(centers, means),
        "L_mean": float(np.mean(ls)) if ls else None,
        "L_p90": float(np.percentile(ls, 90)) if ls else None,
    }
def _load_all(
    model_name: str, domains: list[str], n_samples: int, max_length: int
) -> dict[str, list[dict]]:
    """Per domain: n_samples full sequences, exactly max_length tokens."""
    per: dict[str, list[dict]] = {}
    for dom in domains:
        full = load_calibration_data(
            model_name,
            max_length=max_length,
            min_length=max_length,  # force exactly max_length (truncation active)
            n_samples=n_samples,
            domain=dom,
        )
        if len(full) < n_samples:
            full = load_calibration_data(
                model_name,
                max_length=max_length,
                min_length=1,
                n_samples=n_samples,
                domain=dom,
            )
        per[dom] = full
    return per
def _one_forward(m, s: dict, layers: list[int]) -> dict:
    """ONE forward per sample: full logits + last-3 hidden states per layer.
    v4 fix (colleague 1.2): gauge p must come from the SAME position as J,
    so the full logits tensor is kept; the J pass (backward) reuses this
    forward and does not re-run the model.
    """
    ids = s["input_ids"].to(m.device)
    attn = s["attention_mask"].to(m.device)
    with torch.no_grad():
        out = m(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    logits = out.logits[0].float().cpu()
    hs: dict[int, list[torch.Tensor]] = {}
    for l in layers:
        hs[l] = [out.hidden_states[l + 1][0, i, :].float().cpu() for i in (-1, -2, -3)]
    return {"logits": logits, "hs": hs}
def main_01coherence(
    model="Qwen/Qwen3.5-0.8B",
    device="auto",
    results_dir="results",
    layers=(6, 12, 18, 22),
    K=64,
    n_samples=8,
    domains=("wiki", "math", "code", "chat"),
    max_length=64,
    make_png: bool = True,
    expect_pool_fp: str | None = None,
):
    torch.manual_seed(42)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    from transformers import AutoModelForCausalLM
    print(f"Loading {model}...")
    m = AutoModelForCausalLM.from_pretrained(
        model, torch_dtype=torch.float32, attn_implementation="eager"
    ).to(device)
    m.eval()
    layers = list(layers)
    domains = list(domains)
    print(f"Loading {n_samples} samples x {len(domains)} domains...")
    per_domain = _load_all(model, domains, n_samples, max_length)
    # Аудит v6 (строка R1 очереди): пул берётся из streaming-потока, порядок которого не
    # закреплён, поэтому повторный прогон читает ДРУГИЕ тексты -> другой top-token набор ->
    # другие числа. Измерено на v8 против закоммиченного артефакта: params идентичны, а
    # pk_hash разошёлся (5c12c1749837596b -> 30d82ade4ab705b9), и C_at_sigma_d_b на L6
    # изменился в ~30 раз (0.0326 -> 0.0010). Отпечаток пула теперь печатается и кладётся в
    # params; с --expect-pool-fp подмена данных падает громко вместо тихой замены чисел.
    _h = hashlib.sha256()
    for _dom in domains:
        for _s in per_domain[_dom]:
            _h.update(str(_s["input_ids"].tolist()).encode())
    pool_fp = _h.hexdigest()[:16]
    # Отпечаток по сырым текстам (из закреплённого пула, prism.calibration): не зависит от
    # токенизатора, поэтому сравним между моделями и протоколами.
    pool_fp_text = "|".join(f"{d}:{LAST_POOL_FPS.get(d, '?')}" for d in domains)
    print(f"pool_fp={pool_fp} pool_fp_text={pool_fp_text}", flush=True)
    if expect_pool_fp and expect_pool_fp != pool_fp:
        raise SystemExit(
            f"пул не совпал с пререгистрированным: {pool_fp} != {expect_pool_fp}. "
            "Перегенерация на другом пуле не является повтором этого прогона.")
    # v4: ONE forward per sample, full logits cached (gauge p per position).
    fwd: dict[tuple, dict] = {}
    for dom in domains:
        for si, s in enumerate(per_domain[dom][:n_samples]):
            fwd[(dom, si)] = _one_forward(m, s, layers)
    # Global token set: top-K final-logit tokens across all samples (0.8B).
    counts: dict[int, int] = {}
    for dom in domains:
        for si in range(n_samples):
            p = torch.softmax(fwd[(dom, si)]["logits"][-1], dim=-1)
            top = p.topk(K).indices.tolist()
            for t in top:
                counts[t] = counts.get(t, 0) + 1
    token_set = sorted([t for t, _ in sorted(counts.items(), key=lambda x: -x[1])[:K]])
    print(f"Token set |U| = {len(token_set)}")
    # J-cache: per (domain, sample, layer, pos) -> gauge-free J [K, D] AND
    # raw J (for the three-convention analysis, v5). pos 0=S, 1=S-1, 2=S-2.
    # J at S-pos via compute_jacobian_for_tokens on ids[:, :-pos] (causality:
    # exact). Gauge pk from the SAME position.
    J: dict[tuple, torch.Tensor] = {}
    JRAW: dict[tuple, torch.Tensor] = {}
    PK: dict[tuple, torch.Tensor] = {}
    H: dict[tuple, torch.Tensor] = {}
    done = 0
    total = n_samples * len(domains) * len(layers) * 3
    for dom in domains:
        for si in range(n_samples):
            f = fwd[(dom, si)]
            for l in layers:
                for pos in (0, 1, 2):
                    ids_k = per_domain[dom][si]["input_ids"]
                    mask_k = per_domain[dom][si]["attention_mask"]
                    if pos:
                        ids_k = ids_k[:, :-pos]
                        mask_k = mask_k[:, :-pos]
                    p = torch.softmax(f["logits"][-(pos + 1)], dim=-1)
                    pk = p[token_set]
                    pk = (pk / pk.sum()).to(m.device)
                    Jr = compute_jacobian_for_tokens(
                        m, l, ids_k.to(m.device), mask_k.to(m.device), token_set,
                    ).float()
                    Jp = softmax_jacobian(Jr, pk).float()
                    J[(dom, si, l, pos)] = Jp.cpu()
                    JRAW[(dom, si, l, pos)] = Jr.cpu()
                    PK[(dom, si, l, pos)] = pk.cpu()
                    H[(dom, si, l, pos)] = f["hs"][l][pos]
                    done += 1
            print(f"  {dom} s{si}: {len(layers)}L x 3pos J done ({done}/{total})")
    # Per-layer: separate (a)/(b) curves + sigma_d + split gates + ESG half-split.
    per_layer: dict[str, dict] = {}
    for l in layers:
        res: dict = {}
        # (a) intra-sample adjacent positions: (S-1,S) and (S-2,S-1).
        # v6: L per convention — a_ls (pos-pk), a_ls_same, a_ls_nog (no shared L).
        a_cd: list[tuple[float, float]] = []
        a_cd_same: list[tuple[float, float]] = []
        a_cd_nog: list[tuple[float, float]] = []
        a_ls: list[float] = []
        a_ls_same: list[float] = []
        a_ls_nog: list[float] = []
        for dom in domains:
            for si in range(n_samples):
                for p1, p2 in ((1, 0), (2, 1)):
                    J1, J2 = J[(dom, si, l, p1)], J[(dom, si, l, p2)]
                    H1, H2 = H[(dom, si, l, p1)], H[(dom, si, l, p2)]
                    d = (H1 - H2).norm().item()
                    J1r, J2r = JRAW[(dom, si, l, p1)], JRAW[(dom, si, l, p2)]
                    p1v = PK[(dom, si, l, p1)]  # earlier position's gauge on both
                    J1s = softmax_jacobian(J1r, p1v)
                    J2s = softmax_jacobian(J2r, p1v)
                    a_cd.append((d, _cos(J1, J2)))
                    a_ls.append((J1 - J2).norm().item() / max(d, 1e-12))
                    a_cd_same.append((d, _cos(J1s, J2s)))
                    a_ls_same.append((J1s - J2s).norm().item() / max(d, 1e-12))
                    a_cd_nog.append((d, _cos(J1r, J2r)))
                    a_ls_nog.append((J1r - J2r).norm().item() / max(d, 1e-12))
        res["intra_sample"] = _curve_from_pairs(a_cd, a_ls, 2 * n_samples * len(domains))
        res["intra_sample"]["C_raw"] = [c for _, c in a_cd]  # per-pair, pos-pk
        res["intra_sample"]["C_raw_samepk"] = [c for _, c in a_cd_same]
        res["intra_sample"]["C_raw_nogauge"] = [c for _, c in a_cd_nog]
        res["intra_sample"]["delta_raw"] = [d for d, _ in a_cd]
        res["intra_sample_samepk"] = _curve_from_pairs(a_cd_same, a_ls_same, 2 * n_samples * len(domains))
        res["intra_sample_nogauge"] = _curve_from_pairs(a_cd_nog, a_ls_nog, 2 * n_samples * len(domains))
        # v6: gauge share ||P(p)J||_F^2 / ||J||_F^2 per (sample, layer, pos),
        # and cos(p_t, p_{t+1}) on the token-set-normalized distributions.
        gs_vals, cp_vals = [], []
        for dom in domains:
            for si in range(n_samples):
                for pos in (0, 1, 2):
                    Jr = JRAW[(dom, si, l, pos)]
                    pv = PK[(dom, si, l, pos)]
                    Jp = softmax_jacobian(Jr, pv)
                    gs_vals.append(
                        float(Jp.pow(2).sum() / Jr.pow(2).sum().clamp_min(1e-12))
                    )
                for p1, p2 in ((1, 0), (2, 1)):
                    v1, v2 = PK[(dom, si, l, p1)], PK[(dom, si, l, p2)]
                    cp_vals.append(_cos(v1, v2))
        res["gauge_share_mean"] = float(np.mean(gs_vals))
        # gs_vals order: pos (0,1,2) per sample -> pos0 = gs_vals[0::3] etc.
        res["gauge_share_per_pos"] = {
            f"pos{q}": float(np.mean(gs_vals[q::3])) for q in (0, 1, 2)
        }
        res["cos_p_mean"] = float(np.mean(cp_vals))
        res["cos_p_per_pair"] = cp_vals
        # v6: PK reproducibility — hash of all PK on this layer + top-8 tokens
        # by mean pk (full PK not saved: 64x64x8x4x3 too big for the JSON).
        _ph = hashlib.sha256()
        _pk_sum = torch.zeros(K)
        for dom in domains:
            for si in range(n_samples):
                for pos in (0, 1, 2):
                    _ph.update(PK[(dom, si, l, pos)].numpy().tobytes())
                    _pk_sum += PK[(dom, si, l, pos)]
        res["pk_hash"] = _ph.hexdigest()[:16]
        res["pk_top_token_ids"] = [token_set[i] for i in _pk_sum.topk(8).indices.tolist()]
        # (b) intra-domain: C(n,2) sample pairs within one domain.
        intra = []
        b_cd_pool: list[tuple[float, float]] = []
        for dom in domains:
            jb = [J[(dom, si, l, 0)] for si in range(n_samples)]
            hb = [H[(dom, si, l, 0)] for si in range(n_samples)]
            cd_dom, ls_dom = _pair_stats(jb, hb)
            b_cd_pool.extend(cd_dom)
            intra.append(
                {"domain": dom, **_curve_from_pairs(cd_dom, ls_dom, len(jb))}
            )
        res["intra_domain"] = intra
        res["intra_domain_C_raw"] = [c for _, c in b_cd_pool]  # per-pair, for histogram
        # sigma_d per spec: norm of per-dim std (unbiased=False) of full
        # samples' last-position h, mean over domains.
        stds = []
        for dom in domains:
            hs_d = torch.stack([H[(dom, si, l, 0)] for si in range(n_samples)])
            stds.append(hs_d.std(dim=0, unbiased=False).norm().item())
        res["sigma_d"] = float(np.mean(stds))
        def _c_at(cd: list[tuple[float, float]], x: float) -> float | None:
            centers, means = coh_curve(cd)
            if len(centers) < 2:
                return None
            # аудит v6 (мелочь) + продолжение F2: np.interp ВНЕ диапазона центров
            # молча отдаёт значение краю, то есть «C(σ_d)» на межсэмпловом канале
            # было значением первого бина (σ_d лежит ниже всей поддержки на 26–39 %).
            # Значение оставлено как было (числа карточки не подменяются задним
            # числом), но молчать больше нельзя: рядом пишется поле support-статуса.
            return float(np.interp(x, centers, means))
        def _support(cd: list[tuple[float, float]], x: float) -> str:
            centers, _ = coh_curve(cd)
            if len(centers) < 2:
                return "no_curve"
            if centers[0] <= x <= centers[-1]:
                return "inside"
            return (f"below_support_min(delta>={centers[0]:.2f})"
                    if x < centers[0] else f"above_support_max(delta<={centers[-1]:.2f})")
        # v5: SPLIT gates, THREE gauge conventions. Canonical = between-sample
        # (b) at sigma_d; intra-sample (a) in all three conventions reported
        # alongside (not pooled — 1.1).
        res["C_at_sigma_d_b"] = _c_at(b_cd_pool, res["sigma_d"])
        res["C_at_sigma_d_a"] = _c_at(a_cd, res["sigma_d"])
        res["C_at_sigma_d_a_samepk"] = _c_at(a_cd_same, res["sigma_d"])
        res["C_at_sigma_d_a_nogauge"] = _c_at(a_cd_nog, res["sigma_d"])
        res["C_at_sigma_d_support"] = {
            "b": _support(b_cd_pool, res["sigma_d"]),
            "a_pospk": _support(a_cd, res["sigma_d"]),
            "a_samepk": _support(a_cd_same, res["sigma_d"]),
            "a_raw": _support(a_cd_nog, res["sigma_d"]),
            "note": ("вне поддержки = значение КРАЙНЕГО бина, а не C(σ_d); "
                     "C(δ) убывает ⇒ это нижняя оценка C(σ_d)"),
        }
        bc, bm = coh_curve(b_cd_pool)
        ac, am = coh_curve(a_cd)
        res["gate_curve_b"] = {"delta_centers": bc, "C": bm, "n_pairs": len(b_cd_pool)}
        res["gate_curve_a"] = {"delta_centers": ac, "C": am, "n_pairs": len(a_cd)}
        # (c) cross-domain: all between-domain sample pairs.
        cd_cd: list[tuple[float, float]] = []
        cd_ls: list[float] = []
        n_cross = 0
        for di, d1 in enumerate(domains):
            for d2 in domains[di + 1:]:
                for si in range(n_samples):
                    for sj in range(n_samples):
                        J1, J2 = J[(d1, si, l, 0)], J[(d2, sj, l, 0)]
                        H1, H2 = H[(d1, si, l, 0)], H[(d2, sj, l, 0)]
                        d = (H1 - H2).norm().item()
                        cd_cd.append((d, _cos(J1, J2)))
                        cd_ls.append((J1 - J2).norm().item() / max(d, 1e-12))
                        n_cross += 1
        res["cross_domain"] = _curve_from_pairs(cd_cd, cd_ls, n_cross)
        per_layer[str(l)] = res
        cb = res["C_at_sigma_d_b"]
        ca = res["C_at_sigma_d_a"]
        print(
            f"  L{l}: sigma_d={res['sigma_d']:.3f} "
            f"C_b@sigma_d={cb} C_a@sigma_d={ca}"
        )
    # Save JSON.
    all_results = {
        "layers": per_layer,
        "params": {
            "model": model,
            "n_samples": n_samples,
            "K": K,
            "max_length": max_length,
            "device": device,
            "layers": layers,
            "domains": domains,
                    "min_bin_pairs": MIN_BIN_PAIRS,
            "n_bins": N_BINS,
            "token_set_size": len(token_set),
            "pool_fp": pool_fp,
            "pool_fp_text": pool_fp_text,
        },
    }
    os.makedirs(results_dir, exist_ok=True)
    json_path = os.path.join(results_dir, "exp01_coherence_v6.json")
    with open(json_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Saved to {json_path}")
    if make_png:
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=False)
            for ax, l in zip(axes.flat, layers):
                blk_a = per_layer[str(l)]["intra_sample"]
                ax.plot(blk_a["delta_centers"], blk_a["C"], marker="o", ms=4,
                        label="a: intra-sample (positions)")
                for b in per_layer[str(l)]["intra_domain"]:
                    ax.plot(b["delta_centers"], b["C"], marker="s", ms=4, alpha=0.5)
                blk_c = per_layer[str(l)]["cross_domain"]
                ax.plot(blk_c["delta_centers"], blk_c["C"], marker="^", ms=4, alpha=0.7,
                        label="c: cross-domain")
                sd = per_layer[str(l)]["sigma_d"]
                ax.axvline(sd, color="gray", ls="--", lw=0.8)
                ax.set_title(f"L{l} (sigma_d={sd:.3f})")
                ax.set_xscale("log")
                ax.set_xlabel("|h-h'|")
                ax.set_ylabel("C")
                ax.set_ylim(-0.1, 1.05)
                ax.legend(fontsize=7)
            fig.suptitle("Exp 0.1 v4: Jacobian coherence — SPLIT curves (a) vs (b)")
            fig.tight_layout()
            png_path = os.path.join(results_dir, "exp01_coherence.png")
            fig.savefig(png_path, dpi=110)
            plt.close(fig)
            print(f"Saved to {png_path}")
        except Exception as e:
            print(f"PNG failed (non-fatal): {e}")
    # v5: three-convention summary (no field-character verdict — thresholds
    # below are v3-scale screening labels, recalibrate vs 1/sqrt(D) noise).
    # v6: verdict per layer per convention vs 1/sqrt(D) noise floor (D=hidden
    # dim, 1024 for 0.8B). ORTHOGONAL < 1/sqrt(D); WEAK < 0.15; COHERENT above.
    D = m.config.hidden_size
    NOISE_FLOOR = 1.0 / (D ** 0.5)
    def verdict(c: float | None) -> str:
        if c is None:
            return "n/a"
        if c < NOISE_FLOOR:
            return "ORTHOGONAL"
        if c < 0.15:
            return "WEAK"
        return "COHERENT"
    for l in layers:
        g = per_layer[str(l)]
        g["noise_floor"] = NOISE_FLOOR
        for key, tag in (
            ("C_at_sigma_d_a", "pospk"),
            ("C_at_sigma_d_a_samepk", "samepk"),
            ("C_at_sigma_d_a_nogauge", "raw"),
        ):
            g[f"verdict_{tag}"] = verdict(g[key])
        for key, tag in (("C_at_sigma_d_b", "b"),):
            g[f"verdict_{tag}"] = verdict(g[key])
        cb, ca = g["C_at_sigma_d_b"], g["C_at_sigma_d_a"]
        flag = (
            " [FLAG: C_b > 1.5*C_a]"
            if (cb is not None and ca is not None and cb > 1.5 * ca)
            else ""
        )
        r = lambda x: None if x is None else round(x, 3)
        print(
            f"L{l}: C_b={r(cb)} C_a(pospk)={r(ca)}{flag} "
            f"C_a(samepk)={r(g['C_at_sigma_d_a_samepk'])} "
            f"C_a(raw)={r(g['C_at_sigma_d_a_nogauge'])} "
            f"gauge_share={r(g['gauge_share_mean'])} cos_p={r(g['cos_p_mean'])}"
        )
        print(
            f"     verdict: pospk={g['verdict_pospk']} samepk={g['verdict_samepk']} "
            f"raw={g['verdict_raw']} b={g['verdict_b']} "
            f"(noise_floor=1/sqrt({D})={NOISE_FLOOR:.4f})"
        )
    # аудит v6: verdict'ы и noise_floor появлялись ПОСЛЕ json.dump, то есть в
    # записанном артефакте их не было; дописываем JSON после их расчёта (атомарно)
    tmp = json_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(all_results, f, indent=2)
    os.replace(tmp, json_path)
def _selfcheck() -> None:
    """Model-free check: smooth field C monotone + ell exists."""
    # 1) Smooth field: circle. C(delta) must decrease, ell found.
    D = 16
    n = 40
    ts = torch.linspace(0.0, 3.0, n)
    Js, Hs = [], []
    for t in ts:
        base = torch.stack([torch.cos(t), torch.sin(t)])
        Js.append(base.repeat(D // 2))
        Hs.append(t * torch.ones(D))
    cd, ls = _pair_stats(Js, Hs)
    centers, means = coh_curve(cd)
    assert len(centers) >= 3, f"need >=3 bins, got {len(centers)}"
    assert all(means[i] >= means[i + 1] - 1e-6 for i in range(len(means) - 1)), \
        f"C must decrease monotonically: {[round(x, 4) for x in means]}"
    ell = ell_of(centers, means)
    assert ell is not None and 0 < ell < 5, f"bad ell={ell}"
    print(f"selfcheck OK: ell={ell:.3f}")


In [ ]:
# ---------- run exp01_coherence ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_01coherence)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
kwargs['n_samples'] = 8
kwargs['K'] = 64
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_01coherence(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp01_coherence {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] exp01_coherence: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
